# OT Infrastructure Analysis

**Project:** OT-System-Collector Analytics Suite  
**Author:** Maks V. Zaikin  
**Version:** 1.0.0 (Compatible with Collector v3.1.0)  
**Date:** June 2026  

---

**Executive Summary**
This notebook demonstrates the transformation of raw forensic artifacts and system inventory data, collected from complex industrial environments (ICS/OT), into a structured analytical model.

> **Note:** All hostnames and locations used in this demonstration are synthetic (Star Wars themed) to protect sensitive infrastructure details. *May the Force be with your data.*


## Environment Setup

### INCLUDES

In [ ]:
import io
import pandas as pd
import numpy as np
import os
import re
import json
from datetime import datetime, timedelta
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import xml.etree.ElementTree as ET
import struct
import binascii


!pip install python-evtx

### PANDAS

In [ ]:
# Pandas display settings for forensic analysis
pd.set_option('display.max_colwidth', None)  # Show full paths
pd.set_option('display.max_columns', None)   # Show all features
pd.set_option('future.no_silent_downcasting', True)

### VISUAlISATION

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

### GLOBAL CONFIG

In [ ]:
# Root directory where the 'report_<hostname>' folders are located
BASE_PATH = "reports"

# --- 1. System and Functional Mapping ---
# INSTRUCTIONS: Map unique keywords found in your directory paths 
# to a tuple: (Functional Category, Full System Name).
# Example: 'SCADA_V1': ('Process Control', 'Main SCADA System')
SYSTEM_MAP = {
    'KEYWORD_A': ('Category_Alpha', 'System_Name_A'),
    'KEYWORD_B': ('Category_Beta', 'System_Name_B'),
    # Add your site-specific mappings here
}

# --- 2. Vendor Mapping ---
# INSTRUCTIONS: Map the 'Full System Name' defined above to the actual Vendor.
# This allows for centralized vendor-based risk analysis.
VENDOR_MAP = {
    'System_Name_A': 'Vendor_X',
    'System_Name_B': 'Vendor_Y',
}

# --- 3. Hardware and Path Overrides ---
# INSTRUCTIONS: Use this for generic systems (like HVAC) where the 
# vendor is identified by hardware signatures in the path or hostname.
HARDWARE_VENDOR_OVERRIDES = {
    'HW_BRAND_1': 'Vendor_Z',
    'OEM_MODEL_A': 'OEM_Manufacturer',
}

# --- 4. Data Phase Mapping (Standard for OT-System-Collector v3.1.0) ---
# Maps the 13 collection folders to standardized analytical categories.
DATA_PHASE_MAP = {
    '1_identity': 'identity_and_hardware',
    '2_storage': 'storage_audit',
    '3_network': 'network_topology',
    '4_access': 'access_and_security',
    '5_runtime': 'runtime_and_drivers',
    '6_software': 'software_and_compliance',
    '7_persistence': 'persistence_and_gpo',
    '8_ot_specific': 'ot_hardware',
    '9_forensics': 'event_logs',
    '10_volatile': 'volatile_forensics',
    '11_ot_registry': 'ot_registry',
    '12_remote_access': 'remote_access',
    '13_integrity': 'integrity_markers'
}

# --- 5. Security and ICS Keywords ---
# INSTRUCTIONS: Define keywords to identify critical processes, 
# ICS-related services, and hardware license dongles.
# Users should populate these lists based on their specific environment.

# Keywords for identifying critical system or security processes (e.g., Backup, DB, AV)
CRITICAL_PROCESS_KEYWORDS = ['BACKUP_AGENT', 'SQL_ENGINE', 'AV_SERVICE', 'JAVA_RUNTIME', 'ICS_PROCESS_1']

# Keywords for identifying ICS-related services or vendor-specific software
ICS_SERVICE_KEYWORDS = ['VENDOR_NAME_1', 'VENDOR_NAME_2', 'SCADA_SERVICE', 'PLC_COMM_SVC']

# Keywords for identifying hardware license dongles (e.g., HASP, Sentinel)
LICENSE_DONGLE_KEYWORDS = ['DONGLE_TYPE_1', 'DONGLE_TYPE_2', 'LICENSE_PROTECTOR']

# --- 6. Node Type Mapping ---
# INSTRUCTIONS: Map hostname prefixes or keywords to functional roles.
# Example: 'SRV': 'Server', 'OWS': 'Operator Station'
NODE_TYPE_MAP = {
    'PREFIX_A': 'Server',
    'PREFIX_B': 'Operator Station',
    'PREFIX_C': 'Engineering Workstation',
    'PREFIX_D': 'HMI Panel',
    # Add your specific naming conventions here
}

### GLOBAL FUNCTIONS

In [ ]:
def ultimate_date_parser(raw_val):
    """
    Refined forensic date normalizer. 
    Incorporates logic to discard hex artifacts (starting with '01') 
    and normalizes valid dates to midnight for consistent analytical grouping.
    """
    import re
    
    # Basic cleaning: remove null bytes and whitespace
    s = str(raw_val).replace('\x00', '').strip().lower()
    
    # 1. Discard empty, NaN, or 'Never' values
    if not s or s in ['nan', '0', 'none', 'never']:
        return pd.NaT
    
    # 2. DISCARD HEX ARTIFACTS (The '01' Rule)
    # WMIC often outputs raw FileTime hex strings (e.g., 01d169...) 
    # which are unreliable for automated parsing in legacy environments.
    if s.startswith('01') and len(s) >= 15:
        return pd.NaT

    # 3. Parse standard date strings
    # We try both US (MM/DD) and European (DD/MM) formats
    for day_first in [False, True]:
        try:
            # We use a regex to keep only date-related characters for pandas
            clean_s = re.sub(r'[^0-9/.\-]', '', s)
            d = pd.to_datetime(clean_s, dayfirst=day_first, errors='raise')
            
            # Sanity check: ignore system default dates (1601) or far future
            if 1980 < d.year < 2030:
                return d.normalize() # Truncate to midnight
        except:
            continue
            
    return pd.NaT

def parse_hardware_csv(file_path, file_name):
    """
    UNIVERSAL CSV PARSER for OT-System-Collector.
    Handles all structured CSV outputs from WMIC and Schtasks.
    Supports: CPU, Memory, Product, Services, AV, Tasks, PnP, COM, USB.
    """

    try:
        # 1. Robust Encoding Sniffing & Reading
        content = None
        # Try common Windows encodings
        for enc in ['cp1251', 'utf-16', 'utf-8']:
            try:
                with open(file_path, 'r', encoding=enc) as f:
                    content = f.read()
                if "," in content: break
            except: continue
        
        if not content: return None

        # 2. Sanitization
        # Remove null bytes (common in WMIC) and strip whitespace
        content = content.replace('\x00', '').strip()
        
        # 3. Load into DataFrame
        # StringIO allows pandas to treat the cleaned string as a file
        df_tmp = pd.read_csv(io.StringIO(content), on_bad_lines='skip', engine='python')
        
        # 4. Header Cleaning
        # Remove quotes and spaces from column names
        df_tmp.columns = [c.strip().replace('"', '') for c in df_tmp.columns]
        
        if df_tmp.empty: return None

        fn = file_name.lower()
        
        # --- LOGIC BRANCHES BASED ON FILE NAME ---

        # A. Multi-row Datasets (Returns List of Dicts)
        
        # Scheduled Tasks
        if 'scheduled_tasks' in fn:
            return df_tmp.to_dict('records')

        # Services (WMI Export)
        elif 'services_wmi' in fn:
            return df_tmp.to_dict('records')

        # Antivirus Status
        elif 'av_status' in fn:
            return df_tmp.rename(columns={'displayName': 'av_product_name', 'productState': 'av_product_state'}).to_dict('records')

        # OT-Specific: PnP Devices, COM Ports, USB Controllers
        elif fn in ['pnp', 'com_ports', 'usb_ctrl', 'parallel_ports', 'sound_devices']:
            return df_tmp.to_dict('records')

        # B. Single-host Passport Data (Returns Dictionary)
        
        res = {}
        # CPU Details
        if 'cpu' in fn:
            res['cpu_cores_count'] = len(df_tmp)
            if 'Name' in df_tmp.columns: res['cpu_model'] = str(df_tmp['Name'].iloc[0]).strip()
        
        # RAM / Memory Modules
        elif 'memory' in fn:
            if 'Capacity' in df_tmp.columns:
                total_bytes = pd.to_numeric(df_tmp['Capacity'], errors='coerce').sum()
                res['ram_total_gb'] = round(total_bytes / (1024**3), 2)
        
        # System Product (Serial/Model)
        elif 'csproduct' in fn:
            if 'IdentifyingNumber' in df_tmp.columns: res['serial_number'] = str(df_tmp['IdentifyingNumber'].iloc[0]).strip()
            if 'Name' in df_tmp.columns: res['model'] = str(df_tmp['Name'].iloc[0]).strip()
            if 'Vendor' in df_tmp.columns: res['vendor'] = str(df_tmp['Vendor'].iloc[0]).strip()
            
        return res
    except:
        return None

def rot13_decode(text):
    """Decodes ROT13 encoded strings used in Windows UserAssist registry keys."""
    import codecs
    return codecs.encode(text, 'rot_13')

def parse_registry_forensics(file_path, host_id, mode='userassist'):
    """
    Parses raw 'reg query' output for forensic artifacts.
    Supports: UserAssist (ROT13), BAM/DAM, and ShimCache.
    """
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        
        # Pattern to match registry values and their data
        # Example: {GUID}\path.exe    REG_BINARY    HEX_DATA
        matches = re.findall(r'^\s+(.*?)\s+(REG_SZ|REG_BINARY|REG_DWORD|REG_MULTI_SZ)\s+(.*)$', content, re.MULTILINE)
        
        for m in matches:
            raw_name = m[0].strip()
            raw_data = m[2].strip()
            
            entry = {
                'host_id': host_id,
                'artifact_type': mode.upper(),
                'raw_key_name': raw_name,
                'source_file': file_path
            }
            
            if mode == 'userassist':
                # UserAssist keys are ROT13 encoded
                entry['decoded_name'] = rot13_decode(raw_name)
            elif mode == 'bam':
                # BAM tracks background execution
                entry['decoded_name'] = raw_name
            
            results.append(entry)
        return results
    except:
        return []

def parse_sw_reg_standard(file_path, host_id):
    """Parses standard registry software dumps (DisplayName list)."""
    sw_entries = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
        
        for line in lines:
            sw_entries.append({
                'host_id': host_id,
                'sw_name': line,
                'sw_version': "See Name", # Version is often part of the string
                'sw_architecture': 'x64/Standard',
                'sw_vendor': get_sw_vendor(line),
                'is_ics_related': is_ics_software(line),
                'is_aligned_error': False,
                'source_file': file_path
            })
        return sw_entries, f"SUCCESS: {len(sw_entries)} items"
    except Exception as e:
        return [], f"ERROR: {str(e)}"

def parse_services_sc(file_path, host_id):
    services = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        
        # Split by SERVICE_NAME to get individual blocks
        blocks = content.split('SERVICE_NAME:')
        for block in blocks[1:]: # Skip first empty split
            lines = block.strip().splitlines()
            if not lines: continue
            
            s_name = lines[0].strip()
            data = {'host_id': host_id, 'service_name': s_name, 'source_file': file_path}
            
            for line in lines[1:]:
                if ':' in line:
                    key, val = line.split(':', 1)
                    key = key.strip().upper()
                    val = val.strip()
                    if key == 'DISPLAY_NAME': data['display_name'] = val
                    elif key == 'TYPE': data['type'] = val
                    elif key == 'STATE': data['state'] = val
                    elif key == 'WIN32_EXIT_CODE': data['exit_code'] = val
            
            # ICS Relevance Check
            n = (data.get('display_name', '') + s_name).upper()
            data['is_ics_related'] = any(x in n for x in ['ACRONIS', 'VEEAM', 'SQL', 'SYMANTEC', 'DELTAV', 'ABB', 'SIEMENS'])
            services.append(data)
            
        return services, f"SUCCESS: {len(services)} services"
    except Exception as e:
        return [], f"ERROR: {str(e)}"



## Stage 1: Data Inventory & File Mapping

**Overview**
This stage performs a recursive scan of the base directory to build the **Global File Map (`df_file_map`)**. This registry ensures that every file collected by the script is indexed, categorized by its collection phase, and linked to a specific host using a unique identifier.

Later, parser will built files scan map based on this dataset using pandas queries. That being saing, you should keep in mind if during analysis, you will add new files, program logick won;t see 'em until you re-scan `df_file_map` dataset.

**Key Objectives:**
1. **Recursive File Scanning:** Listing all files collected across the 13 phases.
2. **Host Identification:** Generating a unique `host_id` using the double-underscore format (`Location__System__Hostname`).

   Thus your folders structure should be following: `\reports\location-The_Death_Start\HypermatterReactors\report_HPR_DESCARTES\<scan results>`
   
4. **Phase Mapping:** Categorizing files based on the script's 13-phase directory structure.
5. **Metadata Capture:** Recording file size, extensions, and full paths for data lineage.


### Local Functions

In [ ]:
def get_system_details(path):
    """
    Categorizes the system by scanning the path for keywords defined in SYSTEM_MAP.
    Returns: (Category, System Name)
    """
    path_up = path.upper()
    for key, values in SYSTEM_MAP.items():
        if key.upper() in path_up:
            return values
    return ('General Infrastructure', 'General System')

def get_vendor(system_name, full_path, hostname):
    """
    Identifies the vendor using VENDOR_MAP and HARDWARE_VENDOR_OVERRIDES.
    """
    s = system_name.upper()
    p = full_path.upper()
    h = hostname.upper()

    # 1. Check direct system-to-vendor mapping
    for sys_key, vendor in VENDOR_MAP.items():
        if sys_key.upper() in s:
            return vendor

    # 2. Check hardware-based overrides
    for key, vendor in HARDWARE_VENDOR_OVERRIDES.items():
        if key.upper() in p or key.upper() in h:
            return vendor
    
    return "Generic Industrial"

def get_data_category(path):
    """
    Maps the file path to an analytical category based on the 13-phase structure.
    """
    p_lower = path.lower()
    for folder, category in DATA_PHASE_MAP.items():
        if folder in p_lower:
            return category
    
    # Fallback for non-standard artifacts
    if "images" in p_lower or "pictures" in p_lower: return "visual_evidence"
    if ".pdf" in p_lower or ".md" in p_lower: return "documentation"
    return "uncategorized"

def get_location(path_parts):
    """Extracts site/location name from folders starting with 'location-'."""
    for part in path_parts:
        if "location-" in part.lower():
            return part.replace("location-", "").upper()
    return "UNKNOWN_LOCATION"

def get_hostname(path_parts):
    """Identifies the host name using 'report_' prefix or directory backtracking."""
    for part in path_parts:
        if part.lower().startswith("report_"):
            return part[7:].upper()
    
    if len(path_parts) >= 3:
        parent = path_parts[-2]
        grandparent = path_parts[-3]
        is_category = re.match(r'^\d+_.+', parent) or parent.lower() in ['grouppolicy', 'ieak']
        potential_host = grandparent if is_category else parent
        if "location-" not in potential_host.lower() and potential_host.lower() != "reports":
            return potential_host.upper()
    return "INFRASTRUCTURE_FILE"

def get_node_type(hostname, full_path):
    """
    Classifies the node role based on path metadata and hostname prefixes.
    """
    h = hostname.upper()
    p = full_path.upper()
    
    if h == "NON_HOST_FILE" or h == "INFRASTRUCTURE_FILE":
        return "Infrastructure Component"
    
    # 1. Check path for specific infrastructure roles
    if "FIREWALL" in p: return "Firewall"
    if "GATEWAY" in p: return "Network Gateway"
    if "HMI" in p: return "HMI Panel"
    
    # 2. Check hostname prefixes from NODE_TYPE_MAP
    for prefix, role in NODE_TYPE_MAP.items():
        if h.startswith(prefix):
            return role
            
    # 3. Fallback for identified hosts
    return "Industrial Workstation"

### Main Loop

In [ ]:
file_data = []

In [ ]:
for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        full_path = os.path.join(root, file)
        path_parts = full_path.split(os.sep)
        
        # Metadata extraction
        loc = get_location(path_parts)
        host = get_hostname(path_parts)
        sys_category, sys_name = get_system_details(full_path)
        vendor = get_vendor(sys_name, full_path, host)
        data_cat = get_data_category(full_path)
        
        # NEW: Node Type Classification
        node_type = get_node_type(host, full_path)
        
        file_name_raw, extension = os.path.splitext(file)
        file_size = os.path.getsize(full_path)
        
        # Host ID Generation
        sys_id_part = sys_name.replace(" ", "_").lower()
        host_id = f"{loc}__{sys_id_part}__{host}".lower()
        
        file_data.append({
            'host_id': host_id,
            'location': loc,
            'system_category': sys_category,
            'system': sys_name,
            'vendor': vendor,
            'hostname': host,
            'node_type': node_type, # Column now exists
            'data_category': data_cat,
            'file_name': file_name_raw,
            'extension': extension.lower(),
            'file_size_bytes': file_size,
            'full_path': full_path
        })


In [ ]:
# Create the Global File Map
df_file_map = pd.DataFrame(file_data)


In [ ]:
# Display first few entries for verification
df_file_map.info()

In [ ]:
df_file_map.sample(n=3)

## Stage 2: Data Parsing

**Overview**
This stage focuses on the "Data Resuscitation" process. We transform the raw forensic artifacts, system logs, and command outputs collected during the 13 phases into structured, analysis-ready DataFrames. Each parser is engineered to handle legacy formats (Windows XP/2003), varying encodings (UTF-16, CP1251), and inconsistent table layouts to ensure 100% data retention.

**Key Objectives:**
1. **Normalization:** Converting raw text into standardized data types (Dates, Integers, Booleans).
2. **Error Handling:** Implementing robust logic to manage empty files or "Frankenstein" encodings.
3. **Relational Integrity:** Ensuring every parsed row is linked via the `host_id` for future consolidation.


### Identity and Hardware
---
**Overview**
The "System Passport" serves as the technical baseline for every node in the infrastructure. By consolidating data from `systeminfo.txt`, `system_passport.nfo`, and hardware-specific CSVs, we create a unified profile that defines the operating environment and physical (or virtual) characteristics of the host.

**Target Artifacts:**
* `systeminfo.txt`: Full OS, hardware, and hotfix summary.
* `env_vars.txt`: All current environment variables.
* `path.txt`: Executable search path configuration.
* `system_time.txt`: System clock timestamp at the moment of collection.
* `ie_version.txt`: Internet Explorer version details from the registry.
* `timezone.txt`: Active timezone name and bias settings.
* `servicepack.txt`: Service Pack level, build number, and CSDVersion markers.
* `boot_time.txt`: Last boot time and current system uptime.
* `serials.txt`: BIOS serial number, version, and release date.
* `csproduct.csv`: Vendor, model name, and identifying number (Asset Tag).
* `cpu.csv`: Processor details including cores, speed, and architecture.
* `baseboard.csv`: Motherboard manufacturer, product, and serial number.
* `memory.csv`: Physical memory modules: size, speed, and slot distribution.
* `system_passport.nfo`: Complete MSINFO32/WinMSD hardware and software inventory.

**Output Datasets & Mapping:**

1. **`df_os_hw_final` (The System Passport)**
   * *Primary Sources:* `system_passport.nfo`, `systeminfo.txt`.
   * *Enrichment Sources:* `boot_time.txt`, `serials.txt`, `csproduct.csv`, `cpu.csv`, `baseboard.csv`, `memory.csv`, `ie_version.txt`, `timezone.txt`, `servicepack.txt`, `system_time.txt`, `path.txt`.
2. **`df_env_vars_final` (Environment Variables)**
   * *Source:* `env_vars.txt`.


#### Data Schema


| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_os_hw_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `hostname` | string | Cleaned computer name. |
| | `source_file` | string | Path to the primary file used for this record. |
| | `os_name` | string | Full OS name (e.g., Windows 7 Professional). |
| | `os_version` | string | OS Build and Service Pack details. |
| | `os_install_date` | datetime | Original OS installation date (normalized). |
| | `last_boot_time` | datetime | Last system startup timestamp (normalized). |
| | `vendor` | string | Hardware manufacturer (Dell, HP, VMware, etc.). |
| | `model` | string | System model name. |
| | `serial_number` | string | BIOS Serial or Asset Tag. |
| | `bios_version` | string | BIOS version and release date. |
| | `cpu_model` | string | Processor model and frequency. |
| | `cpu_cores_count` | int | Number of detected CPU cores. |
| | `ram_total_gb` | float | Total physical memory in Gigabytes. |
| | `timezone` | string | Configured system timezone. |
| | `ie_version` | string | Internet Explorer version from registry. |
| | `domain_workgroup` | string | Network membership status. |
| | `path_variable` | string | Full content of the system PATH variable. |
| | `collection_timestamp` | string | Local time when the script was executed. |
| | `service_pack_info` | string | Raw registry markers for SP and Build. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_env_vars_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `hostname` | string | Cleaned computer name. |
| | `source_file` | string | Path to the `env_vars.txt` file. |
| | `variable_name` | string | Name of the environment variable (e.g., TEMP). |
| | `variable_value` | string | Value assigned to the variable. |


#### Local Funcions

In [ ]:
def parse_system_passport_nfo(file_path, host_id):
    """
    Dispatcher for MSINFO32 files. Detects if the file is XML (Legacy XP) 
    or Plain Text (Modern) and routes to the appropriate adapted parser.
    """
    try:
        with open(file_path, 'rb') as f:
            header = f.read(100)
            if b'<?xml' in header or b'<MsInfo' in header:
                return parse_nfo_xml_legacy(file_path, host_id)
            else:
                return parse_nfo_file(file_path, host_id)
    except:
        return None

def parse_nfo_file(file_path, host_id):
    """Adapted text-based MSINFO32 parser."""
    data = {'os_name': None, 'os_version': None, 'os_architecture': None, 
            'vendor': None, 'model': None, 'cpu_model': None, 'ram_total_gb': 0.0, 
            'timezone': None, 'domain_workgroup': None}
    
    content = None
    for enc in ['utf-16', 'cp1251', 'utf-8']:
        try:
            with open(file_path, 'r', encoding=enc) as f:
                lines = f.readlines()
            content = lines; break
        except: continue
    if not content: return None

    current_section = None
    for line in content:
        line = line.strip()
        if line.startswith('[') and line.endswith(']'):
            current_section = line.strip('[]')
            continue
        if current_section == "System Summary":
            parts = line.split('\t')
            if len(parts) >= 2:
                k, v = parts[0].strip(), parts[1].strip()
                if k == "OS Name": data['os_name'] = v
                elif k == "Version": data['os_version'] = v
                elif k == "System Type": data['os_architecture'] = v
                elif k == "System Manufacturer": data['vendor'] = v
                elif k == "System Model": data['model'] = v
                elif k == "Processor": data['cpu_model'] = v
                elif k == "Time Zone": data['timezone'] = v
                elif k == "User Name": data['domain_workgroup'] = v.split('\\')[0]
                elif "Physical Memory" in k and "GB" in v:
                    try: data['ram_total_gb'] = float(v.replace('GB', '').strip())
                    except: pass
    return data

def parse_nfo_xml_legacy(file_path, host_id):
    """Adapted XML-based WinMSD parser for XP/2003."""
    import xml.etree.ElementTree as ET
    raw_content = None
    for enc in ['utf-16', 'utf-8', 'cp1251']:
        try:
            with open(file_path, 'rb') as f:
                raw_content = f.read().decode(enc)
            break
        except: continue
    if not raw_content: return None

    try:
        xml_start = raw_content.find('<?xml')
        if xml_start == -1: xml_start = raw_content.find('<MsInfo')
        root = ET.fromstring(raw_content[xml_start:].strip())
    except: return None

    data = {'os_name': None, 'os_version': None, 'os_architecture': None, 
            'vendor': None, 'model': None, 'cpu_model': None, 'ram_total_gb': 0.0, 
            'timezone': None, 'domain_workgroup': None}

    def find_val(cat_name, item_name):
        for cat in root.iter('Category'):
            if cat.get('name') == cat_name:
                for d in cat.findall('Data'):
                    if d.findtext('Item') == item_name: return d.findtext('Value')
        return None

    data['os_name'] = find_val("System Summary", "OS Name")
    data['os_version'] = find_val("System Summary", "Version")
    data['os_architecture'] = find_val("System Summary", "System Type")
    data['vendor'] = find_val("System Summary", "System Manufacturer")
    data['model'] = find_val("System Summary", "System Model")
    data['cpu_model'] = find_val("System Summary", "Processor")
    data['timezone'] = find_val("System Summary", "Time Zone")
    
    ram_raw = find_val("System Summary", "Total Physical Memory")
    if ram_raw:
        try: data['ram_total_gb'] = round(float(ram_raw.replace(',', '').split()[0]) / 1024, 2)
        except: pass
    return data

def parse_systeminfo_txt(file_path, host_id):
    """Adapted parser for systeminfo.txt (Key: Value)."""
    data = {}
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                if ':' in line:
                    k, v = [x.strip() for x in line.split(':', 1)]
                    if 'Original Install Date' in k: data['os_install_date'] = v
                    elif 'System Boot Time' in k: data['last_boot_time'] = v
                    elif 'Domain' in k: data['domain_workgroup'] = v
        return data
    except: return None

def parse_reg_query_simple(file_path):
    """
    Parses simple 'reg query' outputs. 
    Returns a dictionary of found values.
    """
    results = {}
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        
        # Look for standard reg query pattern: ValueName    REG_TYPE    Value
        matches = re.findall(r'^\s+(.*?)\s+(REG_SZ|REG_DWORD|REG_MULTI_SZ|REG_EXPAND_SZ)\s+(.*)$', content, re.MULTILINE)
        for m in matches:
            results[m[0].strip()] = m[2].strip()
        return results
    except:
        return {}

def parse_wmic_value_format(file_path):
    """
    Parses WMIC output in /value format (Key=Value).
    """
    results = {}
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                if '=' in line:
                    k, v = [x.strip() for x in line.split('=', 1)]
                    if k and v:
                        results[k] = v
        return results
    except:
        return {}

def parse_env_vars(file_path):
    """
    Parses the output of the 'set' command.
    """
    env_dict = {}
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                if '=' in line:
                    k, v = [x.strip() for x in line.split('=', 1)]
                    env_dict[k] = v
        return env_dict
    except:
        return {}

#### Main Loop

In [ ]:
#  Identity & Hardware Execution Loop

results_passports = []
results_env_vars = []


In [ ]:
# Filter df_file_map for the Identity & Hardware category
phase1_targets = df_file_map[df_file_map['data_category'] == 'identity_and_hardware']


In [ ]:
unique_hosts = phase1_targets['host_id'].unique()

In [ ]:
for h_id in tqdm(unique_hosts, desc="Building System Passports"):
    host_files = phase1_targets[phase1_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    passport = {
        'host_id': h_id, 
        'hostname': h_name, 
        'os_name': None, 
        'os_version': None, 
        'os_install_date': None, 
        'last_boot_time': None,
        'vendor': None, 
        'model': None, 
        'serial_number': None, 
        'bios_version': None,
        'cpu_model': None, 
        'cpu_cores_count': None, 
        'ram_total_gb': 0.0,
        'timezone': None, 
        'ie_version': None, 
        'domain_workgroup': None,
        'source_file': None
    }

    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name']
        f_ext = row['extension']
        
        # 1. Primary Passport (.nfo)
        if f_ext == '.nfo':
            nfo_data = parse_system_passport_nfo(f_path, h_id)
            if nfo_data: 
                passport.update(nfo_data)
                passport['source_file'] = f_path # Set primary source

        # 2. Systeminfo.txt
        elif f_name == 'systeminfo':
            si_data = parse_systeminfo_txt(f_path, h_id)
            if si_data:
                for k, v in si_data.items():
                    if not passport.get(k): passport[k] = v
                if not passport['source_file']: passport['source_file'] = f_path

        # 3. Environment Variables
        elif f_name == 'env_vars':
            env_data = parse_env_vars(f_path)
            for k, v in env_data.items():
                results_env_vars.append({
                    'host_id': h_id, 
                    'hostname': h_name,                     
                    'variable_name': k, 
                    'variable_value': v,
                    'source_file': f_path
                })

        # 4. Structured CSVs
        elif f_ext == '.csv':
            csv_data = parse_hardware_csv(f_path, f_name)
            if isinstance(csv_data, dict):
                passport.update(csv_data)

    results_passports.append(passport)

In [ ]:
# --- Create Final Datasets ---
df_os_hw_final = pd.DataFrame(results_passports)
df_env_vars_final = pd.DataFrame(results_env_vars)

df_os_hw_final['source_file'] = df_os_hw_final.pop('source_file')

In [ ]:
# Normalize dates in the passport using the ultimate date parser
for col in ['os_install_date', 'last_boot_time']:
    if col in df_os_hw_final.columns:
        df_os_hw_final[col] = df_os_hw_final[col].apply(ultimate_date_parser)

In [ ]:
# Display the enriched passport
df_os_hw_final.sample(2)

In [ ]:
df_env_vars_final.sample(2)

### Drives Storage and Disks
---
**Overview**
This phase evaluates the physical and logical storage health of the infrastructure. In ICS environments, storage exhaustion is a primary cause of system instability, leading to HMI freezes, database corruption in Historians, and failed backup routines. By parsing logical, physical, and partition-level data, we establish a clear view of disk capacity, utilization, and physical hardware distribution.

**Target Artifacts:**
* `logical.csv`: Logical drive details including letters, file systems, and free space.
* `physical.csv`: Physical disk hardware inventory (Model, Interface, Serial).
* `partitions.csv`: Partition table mapping and boot indicators.

**Output Datasets & Mapping:**

1. **`df_disks_logical_final` (Logical Volumes)**
   * *Source:* `logical.csv`.
2. **`df_disks_physical_final` (Physical Drives)**
   * *Source:* `physical.csv`.
3. **`df_disks_partitions_final` (Partition Table)**
   * *Source:* `partitions.csv`.


#### Data schema



| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_disks_logical_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `hostname` | string | Cleaned computer name. |
| | `source_file` | string | Path to the `logical.csv` file. |
| | `drive_letter` | string | The logical identifier (e.g., C:, D:). |
| | `file_system` | string | Format type (NTFS, FAT32, etc.). |
| | `size_gb` | float | Total capacity of the logical volume in GB. |
| | `freespace_gb` | float | Available space remaining in GB. |
| | `free_space_pct` | float | Calculated health metric: (Free / Total * 100). |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_disks_physical_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `hostname` | string | Cleaned computer name. |
| | `source_file` | string | Path to the `physical.csv` file. |
| | `disk_model` | string | Manufacturer and model of the physical drive. |
| | `interface_type` | string | Connection type (IDE, SATA, SCSI, USB). |
| | `serial_number` | string | Physical disk serial for hardware asset tracking. |
| | `size_gb` | float | Total physical disk capacity in GB. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_disks_partitions_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `hostname` | string | Cleaned computer name. |
| | `source_file` | string | Path to the `partitions.csv` file. |
| | `index` | int | Partition number on the physical drive. |
| | `size_gb` | float | Size of the specific partition in GB. |
| | `type` | string | Partition type (e.g., GPT, MBR, Primary). |
| | `bootable` | boolean | Indicates if the partition contains boot files. |


#### Local Functions

In [ ]:
def parse_disks(file_path, host_id):
    """
    Aggressive WMIC CSV Parser.
    Handles quoted values, null bytes, and BOM to ensure numeric data extraction.
    """
    import io
    import re
    results = []
    try:
        # 1. Read and strip null bytes
        with open(file_path, 'rb') as f:
            raw_bin = f.read().replace(b'\x00', b'')
        
        if not raw_bin: return [], "FAILED: Empty file"

        # 2. Decode with fallback logic
        content = None
        for enc in ['utf-16', 'utf-8', 'cp1251']:
            try:
                content = raw_bin.decode(enc, errors='ignore')
                if 'Node' in content: break
            except: continue
        
        if not content: return [], "FAILED: Decoding error"

        # 3. Load into Pandas with quote handling
        # engine='python' is more robust for inconsistent CSVs
        df_tmp = pd.read_csv(io.StringIO(content), quotechar='"', skipinitialspace=True, on_bad_lines='skip', engine='python')
        
        # 4. Aggressive Header Cleaning
        # Removes quotes, BOM (\ufeff), and whitespace from column names
        df_tmp.columns = [re.sub(r'[^\w]', '', c) for c in df_tmp.columns]
        
        if df_tmp.empty: return [], "EMPTY: No rows found"

        # 5. Data Extraction and Normalization
        for _, row in df_tmp.iterrows():
            # Convert row to dict and clean string values
            item = {k: (str(v).strip().replace('"', '') if pd.notna(v) else None) for k, v in row.to_dict().items()}
            
            # Add mandatory tracking keys
            item['host_id'] = host_id
            item['source_file'] = file_path

            # Normalize Numeric Fields (Size, FreeSpace)
            # We look for keys that contain these words (case-insensitive)
            for key in list(item.keys()):
                if any(word in key.upper() for word in ['SIZE', 'FREESPACE']):
                    val = item.get(key)
                    if val and val != 'None':
                        # Extract only digits to handle any trailing garbage
                        num_match = re.search(r'\d+', val)
                        if num_match:
                            bytes_val = int(num_match.group())
                            # Create a clean normalized key (e.g., size_gb)
                            norm_key = 'size_gb' if 'SIZE' in key.upper() else 'freespace_gb'
                            item[norm_key] = round(bytes_val / (1024**3), 2)
                        else: item[key.lower() + '_gb'] = 0.0
                    else: item[key.lower() + '_gb'] = 0.0
            
            results.append(item)

        return results, f"SUCCESS: {len(results)} items"
    except Exception as e:
        return [], f"ERROR: {str(e)}"

#### Main Loop


In [ ]:
logical_results = []
physical_results = []
partition_results = []

In [ ]:
phase2_targets = df_file_map[df_file_map['data_category'] == 'storage_audit']
unique_hosts = phase2_targets['host_id'].unique()


In [ ]:
unique_hosts

In [ ]:
for h_id in tqdm(unique_hosts, desc="Auditing Storage Systems"):
    host_files = phase2_targets[phase2_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        data, status = parse_disks(f_path, h_id)
        
        if "SUCCESS" in status:
            for item in data:
                # Add common metadata
                item.update({'hostname': h_name, 'source_file': f_path, 'host_id': h_id})
                
                if 'logical' in f_name:
                    size = item.get('size_gb', 0)
                    free = item.get('freespace_gb', 0)
                    item['free_space_pct'] = round((free / size * 100), 2) if size > 0 else 0.0
                    # Ensure DeviceID is captured (it's the drive letter)
                    item['drive_letter'] = item.get('DeviceID') or item.get('Name') or item.get('Caption')
                    logical_results.append(item)
                    
                elif 'physical' in f_name:
                    physical_results.append({
                        'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                        'disk_model': item.get('Model') or item.get('Caption'),
                        'interface_type': item.get('InterfaceType'),
                        'serial_number': item.get('SerialNumber'),
                        'size_gb': item.get('size_gb', 0)
                    })
                    
                elif 'partitions' in f_name:
                    partition_results.append({
                        'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                        'index': item.get('Index'),
                        'size_gb': item.get('size_gb', 0),
                        'type': item.get('Type'),
                        'bootable': str(item.get('Bootable')).upper() == 'TRUE'
                    })

In [ ]:
df_disks_logical_final = pd.DataFrame(logical_results)
df_disks_physical_final = pd.DataFrame(physical_results)
df_disks_partitions_final = pd.DataFrame(partition_results)

### Network Topology
---
**Overview**
This phase reconstructs the logical and physical network architecture of the ICS environment. By consolidating data from interface configurations, routing tables, ARP caches, and active connections, we build a "Digital Twin" of the network. This is essential for identifying segmentation violations, unauthorized bridges between IT and OT, and mapping the potential lateral movement paths of an attacker.

**Target Artifacts:**
* `ipconfig.txt`: Detailed IP configuration for all adapters.
* `routes.txt` & `routes_num.txt`: Active IPv4/IPv6 routing tables.
* `arp.txt`: IP-to-MAC address resolution cache (neighbor discovery).
* `mac_addresses.txt`: Physical addresses mapped to adapter names.
* `connections.txt`: Active TCP/UDP connections and listening ports (netstat).
* `shares_local.txt`: SMB shares hosted on the machine.
* `shares_mapped.txt`: Remote network drives connected to the host.
* `sessions.txt`: Active inbound SMB sessions.
* `dns_cache.txt`: Recently resolved hostnames.
* `hosts_file.txt`: Static hostname-to-IP overrides.
* `net_config_ws.txt` & `net_config_srv.txt`: Workstation and Server service parameters.
* `tcpip_registry.txt` & `interfaces_registry.txt`: Deep TCP/IP stack settings from the registry.
* `nics.csv`, `nic_config.csv`, `pci_adapters.csv`: Structured WMI data for network hardware.

**Output Datasets & Mapping:**

1. **`df_net_interfaces_final`**
   * *Sources:* `ipconfig.txt`, `nic_config.csv`, `mac_addresses.txt`.
2. **`df_net_routes_final`**
   * *Sources:* `routes.txt`, `routes_num.txt`.
3. **`df_net_arp_final`**
   * *Source:* `arp.txt`.
4. **`df_net_connections_final`**
   * *Source:* `connections.txt`.
5. **`df_net_shares_final`**
   * *Sources:* `shares_local.txt`, `shares_mapped.txt`.
6. **`df_net_config_final`**
   * *Sources:* `net_config_ws.txt`, `net_config_srv.txt`, `hosts_file.txt`, `dns_cache.txt`.



#### Data Schema

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_net_interfaces_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `adapter_name` | string | Name of the network interface. |
| | `ipv4_address` | string | Assigned IPv4 address. |
| | `mac_address` | string | Physical (MAC) address. |
| | `gateway` | string | Default gateway for the interface. |
| | `dhcp_enabled` | boolean | Indicates if IP is assigned via DHCP. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_net_connections_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `protocol` | string | TCP or UDP. |
| | `local_address` | string | Local IP and Port. |
| | `remote_address` | string | Remote IP and Port. |
| | `state` | string | Connection state (LISTENING, ESTABLISHED, etc.). |
| | `pid` | int | Process ID owning the connection. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_net_arp_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `local_interface` | string | The IP of the interface that saw the neighbor. |
| | `remote_ip` | string | Neighbor's IP address. |
| | `physical_address` | string | Neighbor's MAC address. |
| | `type` | string | Dynamic or Static entry. |



#### Local Functions

In [ ]:
def parse_ipconfig_v3(file_path, host_id):
    """
    Forensic-grade ipconfig parser. 
    Captures all adapters regardless of connection state.
    """
    results = []
    current = None
    
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                clean = line.strip()
                if not clean or "Windows IP Configuration" in clean:
                    continue
                
                # 1. Detect Header: No leading whitespace and ends with ':'
                # We exclude lines that look like global config (Host Name, etc.)
                if not line.startswith((' ', '\t')) and clean.endswith(':'):
                    if current:
                        results.append(current)
                    
                    current = {
                        'host_id': host_id, 
                        'adapter_name': clean[:-1].strip(), 
                        'source_file': file_path, 
                        'dns_servers': [],
                        'description': None, 'mac_address': None, 'ipv4_address': None,
                        'subnet_mask': None, 'gateway': None, 'is_dhcp': None
                    }
                    continue
                
                # 2. Detect Data: Contains a colon
                if current and ':' in line:
                    parts = line.split(':', 1)
                    # Clean key: remove dots and lowercase for stable matching
                    k = parts[0].replace('.', '').strip().lower()
                    v = parts[1].strip()
                    
                    if 'description' in k: current['description'] = v
                    elif 'physical address' in k: current['mac_address'] = v.lower().replace('-', ':')
                    elif 'ipv4 address' in k: current['ipv4_address'] = v.replace('(Preferred)', '').strip()
                    elif 'subnet mask' in k: current['subnet_mask'] = v
                    elif 'default gateway' in k: current['gateway'] = v if v else None
                    elif 'dhcp enabled' in k: current['is_dhcp'] = v.lower() == 'yes'
                    elif 'dns servers' in k:
                        if v: current['dns_servers'].append(v)
                
                # 3. Handle Multi-line DNS servers
                elif current and line.startswith((' ' * 20, '\t' * 4)) and clean and '.' in clean:
                    # Ensure it's not another property line
                    if ':' not in clean:
                        current['dns_servers'].append(clean)
            
            # Append the last adapter found in file
            if current:
                results.append(current)
        
        # Finalize: Convert DNS lists to JSON strings
        for item in results:
            item['dns_servers'] = json.dumps(item['dns_servers'])
            
        return results
    except:
        return []

def parse_routes_v3(file_path, host_id):
    """Parses route print output for IPv4 tables."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        # Identify Active vs Persistent
        sections = re.split(r'(Active Routes:|Persistent Routes:)', content)
        for i in range(1, len(sections), 2):
            r_type = 'Active' if 'Active' in sections[i] else 'Persistent'
            table = sections[i+1]
            # Match IPv4 rows: Dest, Mask, Gateway, Interface, Metric
            matches = re.findall(r'^\s*([\d\.]+)\s+([\d\.]+)\s+(\S+)\s+([\d\.]+)\s+(\d+)', table, re.MULTILINE)
            for m in matches:
                results.append({
                    'host_id': host_id, 'destination': m[0], 'netmask': m[1], 'gateway': m[2],
                    'interface': m[3], 'metric': int(m[4]), 'route_type': r_type, 'source_file': file_path
                })
        return results
    except: return []

def parse_netstat_v3(file_path, host_id):
    """Parses netstat -aon output using regex."""
    results = []
    pattern = r'^\s*(?P<proto>TCP|UDP)\s+(?P<local>\S+)\s+(?P<remote>\S+)\s+(?P<state>\w+)?\s+(?P<pid>\d+)$'
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                m = re.match(pattern, line.strip())
                if m:
                    d = m.groupdict()
                    results.append({
                        'host_id': host_id, 'protocol': d['proto'], 'local_address': d['local'],
                        'remote_address': d['remote'], 'state': d['state'] or 'UDP', 
                        'pid': int(d['pid']), 'source_file': file_path
                    })
        return results
    except: return []

def parse_dns_cache(file_path, host_id):
    """
    Refined DNS cache parser.
    Uses flexible regex to handle variable dot-leaders and whitespace.
    """
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
            
        # Split by the standard DNS record separator
        blocks = content.split('----------------------------------------')
        
        for block in blocks:
            # Flexible regex: matches "Key . . . : Value" with any number of dots/spaces
            name_m = re.search(r'Record Name\s*[\.\s]*:\s*(\S+)', block, re.I)
            type_m = re.search(r'Record Type\s*[\.\s]*:\s*(\d+)', block, re.I)
            
            # Look for the actual resolution (A record, CNAME, or PTR)
            val_m = re.search(r'(?:A \(Host\) Record|CNAME Record|PTR Record)\s*[\.\s]*:\s*(\S+)', block, re.I)
            
            if name_m and type_m:
                results.append({
                    'host_id': host_id,
                    'record_name': name_m.group(1),
                    'record_type': type_m.group(1),
                    'resolved_address': val_m.group(1) if val_m else None,
                    'source_type': 'DNS_CACHE',
                    'source_file': file_path
                })
        return results
    except: return []
def parse_reg_interfaces(file_path, host_id):
    """Parses registry dump of TCP/IP interfaces."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        # Split by GUID blocks
        blocks = re.split(r'HKEY_LOCAL_MACHINE\\.*?\\Interfaces\\', content)
        for block in blocks[1:]:
            guid_match = re.match(r'\{[A-F0-9-]+\}', block, re.I)
            if guid_match:
                guid = guid_match.group(0)
                ip_m = re.search(r'IPAddress\s+REG_MULTI_SZ\s+(\S+)', block)
                dhcp_m = re.search(r'EnableDHCP\s+REG_DWORD\s+(0x[01])', block)
                results.append({
                    'host_id': host_id, 'interface_guid': guid, 
                    'reg_ip': ip_m.group(1) if ip_m else None,
                    'is_dhcp_enabled': dhcp_m.group(1) == '0x1' if dhcp_m else None,
                    'source_file': file_path
                })
        return results
    except: return []

def parse_net_config(file_path, host_id):
    """Parses net config workstation/server output."""
    res = {'host_id': host_id, 'source_file': file_path}
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                if ':' in line: # Server format
                    k, v = line.split(':', 1)
                else: # Workstation format (fixed width)
                    k, v = line[:30], line[30:]
                k, v = k.strip(), v.strip()
                if 'Workstation domain' in k: res['workstation_domain'] = v
                elif 'Logon domain' in k: res['logon_domain'] = v
                elif 'Software version' in k: res['software_version'] = v
        return res
    except: return {}

def parse_hosts_file(file_path, host_id):
    """Parses standard Windows hosts file."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                clean = line.strip()
                if not clean or clean.startswith('#'): continue
                parts = clean.split()
                if len(parts) >= 2:
                    results.append({
                        'host_id': host_id, 'record_name': parts[1], 
                        'record_type': '1 (Static)', 'resolved_address': parts[0],
                        'source_type': 'HOSTS_FILE', 'source_file': file_path
                    })
        return results
    except: return []

def parse_arp_file(file_path, host_id):
    """
    Parses 'arp -a' output. Handles multiple local interfaces per file.
    Returns (list of dicts, status_message)
    """
    arp_entries = []
    current_interface = None
    try:
        if os.path.getsize(file_path) == 0:
            return [], "EMPTY: File size is 0"

        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            lines = f.readlines()
            
        for line in lines:
            line = line.strip()
            if not line: continue
            
            # Detect local interface block
            if "Interface:" in line:
                match = re.search(r'Interface:\s+([\d\.]+)', line)
                if match:
                    current_interface = match.group(1)
                continue
            
            # Detect data rows (IP, MAC, Type)
            # Pattern: 192.168.10.1  00-11-32-53-79-56  dynamic
            row_match = re.search(r'([\d\.]+)\s+([0-9a-fA-F-]{17})\s+(\w+)', line)
            if row_match and current_interface:
                arp_entries.append({
                    'host_id': host_id,
                    'local_interface': current_interface,
                    'remote_ip': row_match.group(1),
                    'physical_address': row_match.group(2).lower().replace('-', ':'),
                    'arp_type': row_match.group(3),
                    'source_file': file_path
                })
        
        if not arp_entries:
            return [], "FAILED: No ARP entries found"
            
        return arp_entries, f"SUCCESS: {len(arp_entries)} entries"
    except Exception as e:
        return [], f"ERROR: {str(e)}"

def parse_net_shares(file_path, host_id):
    """
    Parses 'NET SHARE' output using greedy whitespace splitting.
    Handles multi-line entries and long share names.
    """
    shares = []
    try:
        if os.path.getsize(file_path) == 0:
            return [], "EMPTY: File size is 0"

        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            lines = f.readlines()

        sep_idx = -1
        for i, line in enumerate(lines):
            if line.startswith('---'):
                sep_idx = i
                break
        if sep_idx == -1: return [], "FAILED: Separator not found"

        current_share = None
        for line in lines[sep_idx+1:]:
            line_raw = line.rstrip('\n\r')
            if not line_raw.strip() or "The command completed successfully" in line_raw:
                continue
            
            if len(line_raw) > 0 and not line_raw.startswith(' '):
                if current_share: shares.append(current_share)
                parts = re.split(r'\s{2,}', line_raw.strip())
                current_share = {
                    'host_id': host_id,
                    'share_name': parts[0] if len(parts) > 0 else "",
                    'resource_path': parts[1] if len(parts) > 1 else "",
                    'remark': parts[2] if len(parts) > 2 else "",
                    'source_file': file_path
                }
            elif current_share and line_raw.startswith(' '):
                sub_parts = re.split(r'\s{2,}', line_raw.strip())
                for p in sub_parts:
                    p = p.strip()
                    if not p: continue
                    if (':' in p and '\\' in p) or p.startswith('\\\\'):
                        current_share['resource_path'] = (current_share['resource_path'] + p).strip()
                    else:
                        current_share['remark'] = (current_share['remark'] + " " + p).strip()

        if current_share: shares.append(current_share)
        return shares, f"SUCCESS: {len(shares)} shares"
    except Exception as e:
        return [], f"ERROR: {str(e)}"

#### Main Loop

In [ ]:
# Containers for multiple network-related datasets
net_ifaces, net_routes, net_arp, net_conns, net_shares, net_dns, net_config = [], [], [], [], [], [], []

# Filter df_file_map for Network Topology phase
phase3_targets = df_file_map[df_file_map['data_category'] == 'network_topology']


In [ ]:
unique_hosts = phase3_targets['host_id'].unique()


In [ ]:
for h_id in tqdm(unique_hosts, desc="Reconstructing Network Topology"):
    host_files = phase3_targets[phase3_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path, f_name = row['full_path'], row['file_name'].lower()
        
        if f_name == 'ipconfig': net_ifaces.extend(parse_ipconfig_v3(f_path, h_id))
        elif 'routes' in f_name: net_routes.extend(parse_routes_v3(f_path, h_id))
        elif f_name == 'arp': net_arp.extend(parse_arp_file(f_path, h_id)[0])
        elif f_name == 'connections': net_conns.extend(parse_netstat_v3(f_path, h_id))
        elif 'shares' in f_name: net_shares.extend(parse_net_shares(f_path, h_id)[0])
        elif f_name == 'dns_cache': net_dns.extend(parse_dns_cache(f_path, h_id))
        elif f_name == 'hosts_file': net_dns.extend(parse_hosts_file(f_path, h_id))
        elif 'net_config' in f_name or 'tcpip_registry' in f_name: 
            net_config.append(parse_net_config(f_path, h_id))
        elif 'interfaces_registry' in f_name:
            # This could be merged into net_config or kept separate
            pass 

In [ ]:
# --- Create Final Network Datasets ---
df_net_interfaces_final = pd.DataFrame(net_ifaces)
df_net_routes_final = pd.DataFrame(net_routes)
df_net_arp_final = pd.DataFrame(net_arp)
df_net_connections_final = pd.DataFrame(net_conns)
df_net_shares_final = pd.DataFrame(net_shares)
df_net_dns_final = pd.DataFrame(net_dns)
df_net_config_final = pd.DataFrame(net_config)


### Access & Security
---
**Overview**
This phase audits the identity and access management (IAM) layer of the infrastructure. By analyzing local user accounts, group memberships, and password policies, we identify potential security weaknesses such as over-privileged accounts, stale users, or weak password requirements. This data is essential for mapping the "blast radius" of a compromised credential and ensuring compliance with the Principle of Least Privilege (PoLP).

**Target Artifacts:**
* `users.txt`: Basic list of all local user accounts.
* `users_detail.txt`: Comprehensive per-user audit (last logon, password expiry, group memberships).
* `groups.txt`: Inventory of all local security groups.
* `group_administrators.txt`: Specific membership list for the high-privilege Administrators group.
* `group_rdp_users.txt`: Membership list for the Remote Desktop Users group.
* `pwd_policy.txt`: Global password complexity and lockout requirements.
* `wmi_users.csv`: Structured WMI data including SIDs and account status flags.

**Output Datasets & Mapping:**

1. **`df_users_final` (User Registry)**
   * *Sources:* `wmi_users.csv`, `users.txt`.
2. **`df_users_detailed_final` (Deep Account Audit)**
   * *Source:* `users_detail.txt`.
3. **`df_groups_final` (Security Groups)**
   * *Source:* `groups.txt`.
4. **`df_group_members_final` (Privileged Memberships)**
   * *Sources:* `group_administrators.txt`, `group_rdp_users.txt`.
5. **`df_pwd_policy_final` (Security Baseline)**
   * *Source:* `pwd_policy.txt`.



#### Data Schema


| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_users_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `user_name` | string | Account name. |
| | `sid` | string | Security Identifier (the "digital fingerprint" of the account). |
| | `is_disabled` | boolean | Account status flag. |
| | `is_local` | boolean | True if the account is local to the machine. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_users_detailed_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `user_name` | string | Account name. |
| | `last_logon` | datetime | Timestamp of the last successful entry. |
| | `password_never_expires` | boolean | High-risk flag for service or neglected accounts. |
| | `member_of_json` | list | JSON array of all groups the user belongs to. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_group_members_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `group_name` | string | Name of the privileged group (e.g., Administrators). |
| | `member_name` | string | Name of the user or group assigned to this role. |


#### Local Functions

In [ ]:
import json
import re
import pandas as pd

def get_account_type_str(at_id):
    mapping = {'512': 'Normal Account', '256': 'Temp Duplicate', '128': 'Interdomain Trust', '64': 'Workstation Trust', '32': 'Server Trust'}
    return mapping.get(str(at_id), f"Unknown ({at_id})")

def get_sid_type_str(st_id):
    mapping = {'1': 'User', '2': 'Group', '3': 'Domain', '4': 'Alias', '5': 'Well-known Group', '9': 'Computer'}
    return mapping.get(str(st_id), f"Unknown ({st_id})")

def parse_userlist_detailed(file_path, host_id):
    """Parses 'net user' output using predefined keywords as anchors."""
    users_data = []
    anchors = [
        "User name", "Full Name", "Comment", "User's comment", 
        "Country/region code", "Account active", "Account expires", 
        "Password last set", "Password expires", "Password changeable", 
        "Password required", "User may change password", "Workstations allowed", 
        "Logon script", "User profile", "Home directory", "Last logon", 
        "Logon hours allowed", "Local Group Memberships", "Global Group memberships"
    ]
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        blocks = content.split('The command completed successfully.')
        for block in blocks:
            if "User name" not in block: continue
            raw_values = {}
            positions = []
            for a in anchors:
                pos = block.find(a)
                if pos != -1: positions.append((pos, a))
            positions.sort()
            for i in range(len(positions)):
                curr_pos, curr_anchor = positions[i]
                next_pos = positions[i+1][0] if i+1 < len(positions) else len(block)
                val = block[curr_pos + len(curr_anchor):next_pos].strip().lstrip(':').strip()
                raw_values[curr_anchor] = val
            
            users_data.append({
                'host_id': host_id,
                'user_name': raw_values.get("User name"),
                'full_name': raw_values.get("Full Name"),
                'is_active': raw_values.get("Account active") == "Yes",
                'last_logon': raw_values.get("Last logon"),
                'password_never_expires': "never" in str(raw_values.get("Password expires")).lower(),
                'local_groups': json.dumps(re.findall(r'\*(\S+)', raw_values.get("Local Group Memberships", ""))),
                'source_file': file_path
            })
        return users_data, f"SUCCESS: {len(users_data)} users"
    except Exception as e: return [], f"ERROR: {str(e)}"

def parse_localgroups(file_path, host_id):
    """Parses 'net localgroup' output to extract group names."""
    groups_data = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            lines = f.readlines()
        for line in lines:
            if line.startswith("*"):
                g_name = line.lstrip("*").strip()
                groups_data.append({
                    'host_id': host_id,
                    'group_name': g_name,
                    'source_file': file_path
                })
        return groups_data, f"SUCCESS: {len(groups_data)} groups"
    except Exception as e: return [], f"ERROR: {str(e)}"

def parse_users_wmi(file_path, host_id):
    """
    Updated WMIC CSV parser for user accounts (SIDs and Flags).
    Handles quoted CSV format from OT-System-Collector v3.1.0.
    """
    import io
    processed_users = []
    try:
        # 1. Read and clean null bytes/BOM
        with open(file_path, 'rb') as f:
            raw_bin = f.read().replace(b'\x00', b'')
        
        content = None
        for enc in ['utf-16', 'utf-8', 'cp1251']:
            try:
                content = raw_bin.decode(enc, errors='ignore')
                if 'AccountType' in content: break
            except: continue
            
        if not content: return [], "FAILED: Decoding error"

        # 2. Load as CSV
        df_tmp = pd.read_csv(io.StringIO(content), quotechar='"', skipinitialspace=True, on_bad_lines='skip', engine='python')
        
        # Clean headers (remove non-word characters like BOM)
        df_tmp.columns = [re.sub(r'[^\w]', '', c) for c in df_tmp.columns]
        
        if df_tmp.empty: return [], "EMPTY: No rows found"

        # 3. Process Rows
        for _, row in df_tmp.iterrows():
            u = {k: (str(v).strip() if pd.notna(v) else "") for k, v in row.to_dict().items()}
            
            sid = u.get('SID', '')
            rid = sid.split('-')[-1] if '-' in sid else None
            
            processed_users.append({
                'host_id': host_id,
                'user_name': u.get('Name'),
                'full_name': u.get('FullName'),
                'sid': sid,
                'rid': int(rid) if rid and rid.isdigit() else None,
                'is_disabled': u.get('Disabled', '').upper() == 'TRUE',
                'is_local': u.get('LocalAccount', '').upper() == 'TRUE',
                'account_type': get_account_type_str(u.get('AccountType')),
                'sid_type': get_sid_type_str(u.get('SIDType')),
                'status': u.get('Status'),
                'source_file': file_path
            })
            
        return processed_users, f"SUCCESS: {len(processed_users)} users"
    except Exception as e: 
        return [], f"ERROR: {str(e)}"

def parse_group_members_list(file_path, host_id):
    """Parses 'net localgroup <name>' output for specific members."""
    members = []
    group_name = "Unknown"
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            lines = f.readlines()
        for line in lines:
            if "Alias name" in line: group_name = line.split("Alias name", 1)[1].strip()
            if line.startswith("---"):
                start_idx = lines.index(line) + 1
                for m_line in lines[start_idx:]:
                    m_clean = m_line.strip()
                    if m_clean and "The command completed successfully" not in m_clean:
                        members.append({'host_id': host_id, 'group_name': group_name, 'member_name': m_clean, 'source_file': file_path})
                break
        return members
    except: return []

def parse_net_accounts(file_path, host_id):
    """Parses 'net accounts' output for password policies."""
    res = {'host_id': host_id, 'source_file': file_path}
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                if ':' in line:
                    k, v = line.split(':', 1)
                    res[k.strip().replace('?', '')] = v.strip()
        return res
    except: return {}

#### Main Loop

In [ ]:
# Access and Security Execution Loop ---

# Containers for identity and security datasets
wmi_users_results = []
detailed_users_results = []
local_groups_results = []
group_members_results = []
pwd_policy_results = []


In [ ]:
# Filter df_file_map for Access & Security phase
phase4_targets = df_file_map[df_file_map['data_category'] == 'access_and_security']


In [ ]:
unique_hosts = phase4_targets['host_id'].unique()


In [ ]:
for h_id in tqdm(unique_hosts, desc="Auditing Access & Security"):
    host_files = phase4_targets[phase4_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        # 1. WMI Users (SIDs and Flags)
        if 'wmi_users' in f_name:
            data, status = parse_users_wmi(f_path, h_id)
            if "SUCCESS" in status:
                for item in data:
                    item['hostname'] = h_name
                    wmi_users_results.append(item)

        # 2. Detailed Users (Logon dates, Group memberships)
        elif 'users_detail' in f_name:
            data, status = parse_userlist_detailed(f_path, h_id)
            if "SUCCESS" in status:
                for item in data:
                    item['hostname'] = h_name
                    detailed_users_results.append(item)

        # 3. Local Groups Inventory
        elif f_name == 'groups':
            data, status = parse_localgroups(f_path, h_id)
            if "SUCCESS" in status:
                for item in data:
                    item['hostname'] = h_name
                    local_groups_results.append(item)

        # 4. Specific Group Members (Admins, RDP)
        elif 'group_' in f_name:
            data = parse_group_members_list(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                group_members_results.append(item)

        # 5. Password Policy
        elif 'pwd_policy' in f_name:
            data = parse_net_accounts(f_path, h_id)
            if data:
                data['hostname'] = h_name
                pwd_policy_results.append(data)

In [ ]:
# --- Create Final Access Datasets ---
df_users_wmi_final = pd.DataFrame(wmi_users_results)
df_users_detailed_final = pd.DataFrame(detailed_users_results)
df_local_groups_final = pd.DataFrame(local_groups_results)
df_group_members_final = pd.DataFrame(group_members_results)
df_pwd_policy_final = pd.DataFrame(pwd_policy_results)

### Runtime State and Drivers
---
**Overview**
This phase captures the "living" state of the infrastructure. By analyzing running processes, active services, and loaded kernel drivers, we establish a baseline of normal operational behavior. This data is critical for detecting unauthorized software, identifying service misconfigurations (e.g., services running under high-privilege accounts), and auditing system time synchronization (NTP), which is vital for log integrity across the ICS environment.

**Target Artifacts:**
* `tasks.csv`: Snapshot of all running processes with user context and window titles.
* `services_all.txt`: Current state and type of all registered Windows services.
* `service_paths.txt`: Detailed configuration of each service, including full binary paths and start types.
* `services_wmi.csv`: Structured WMI data for services (PathName, StartMode, StartName).
* `drivers_list.csv`: Comprehensive list of installed drivers with their execution state and file paths.
* `drivers_kernel.txt`: Kernel-mode drivers currently registered or loaded.
* `ntp_status.txt`: Network Time Protocol (NTP) configuration and synchronization health.
* `open_files.txt`: List of files currently opened or locked via network shares.

**Output Datasets & Mapping:**

1. **`df_tasks_final` (Running Processes)**
   * *Source:* `tasks.csv`.
2. **`df_services_final` (Service Inventory)**
   * *Sources:* `services_all.txt`, `service_paths.txt`, `services_wmi.csv`.
3. **`df_drivers_final` (Driver Inventory)**
   * *Sources:* `drivers_list.csv`, `drivers_kernel.txt`.
4. **`df_ntp_final` (Time Sync Status)**
   * *Source:* `ntp_status.txt`.
5. **`df_open_files_final` (File Locks)**
   * *Source:* `open_files.txt`.


#### Data schema



| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_tasks_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `image_name` | string | Executable file name (e.g., `sqlservr.exe`). |
| | `pid` | int | Process ID. |
| | `user_name` | string | Account context under which the process is running. |
| | `mem_usage_kb` | int | Memory footprint in Kilobytes. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_services_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `service_name` | string | Internal system name of the service. |
| | `display_name` | string | Human-readable service name. |
| | `state` | string | Current status (RUNNING, STOPPED, etc.). |
| | `binary_path` | string | Full path to the executable file. |
| | `start_name` | string | Account used to launch the service (e.g., LocalSystem). |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_drivers_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `module_name` | string | Driver module name. |
| | `display_name` | string | Human-readable driver name. |
| | `driver_type` | string | Kernel Driver or File System Driver. |
| | `path` | string | Full path to the `.sys` file. |

#### Local Functions

In [ ]:
def parse_phase5_csv(file_path, host_id):
    """Universal robust CSV reader for Phase 5 artifacts."""
    try:
        with open(file_path, 'rb') as f:
            raw = f.read().replace(b'\x00', b'')
        content = None
        for enc in ['utf-16', 'cp1251', 'utf-8']:
            try:
                content = raw.decode(enc, errors='ignore')
                if ',' in content: break
            except: continue
        if not content: return []
        df_tmp = pd.read_csv(io.StringIO(content), quotechar='"', skipinitialspace=True, on_bad_lines='skip', engine='python')
        df_tmp.columns = [re.sub(r'[^\w]', '', c).replace(' ', '_').lower() for c in df_tmp.columns]
        return df_tmp.to_dict('records')
    except: return []

def parse_tasklist_enriched(file_path, host_id):
    """Parses tasks.csv with performance and context enrichment."""
    results = []
    raw_data = parse_phase5_csv(file_path, host_id)
    for r in raw_data:
        img_name = str(r.get('imagename', '')).upper()
        # 1. Convert CPU Time (HH:MM:SS) to total seconds
        cpu_raw = str(r.get('cputime', '0:00:00'))
        try:
            h, m, s = map(int, cpu_raw.split(':'))
            cpu_sec = h * 3600 + m * 60 + s
        except: cpu_sec = 0
        
        # 2. Normalize Memory
        mem_raw = str(r.get('memusage', '0')).replace(',', '').replace(' K', '').strip()
        
        # 3. Identify System Accounts
        user = str(r.get('username', '')).upper()
        is_sys_acc = any(x in user for x in ['SYSTEM', 'LOCAL SERVICE', 'NETWORK SERVICE'])
        
        results.append({
            'host_id': host_id, 'source_file': file_path,
            'image_name': r.get('imagename'),
            'pid': int(r.get('pid', 0)),
            'session_name': r.get('sessionname'),
            'mem_usage_kb': int(mem_raw) if mem_raw.isdigit() else 0,
            'user_name': r.get('username'),
            'is_system_account': is_sys_acc,
            'cpu_time_total_sec': cpu_sec,
            'window_title': r.get('windowtitle'),
            'is_critical_proc': any(k.upper() in img_name for k in CRITICAL_PROCESS_KEYWORDS)
        })
    return results

def parse_sc_comprehensive(file_path, host_id):
    """Advanced block parser for sc query/qc with dependency support."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        
        blocks = content.split('SERVICE_NAME:')
        for block in blocks[1:]:
            lines = block.splitlines()
            if not lines: continue
            
            s_name = lines[0].strip()
            data = {'host_id': host_id, 'service_name': s_name, 'source_file': file_path, 'dependencies': []}
            
            # Track if we are currently reading a multi-line dependency list
            in_dependencies = False
            
            for line in lines[1:]:
                if ':' in line:
                    # Check if it's a new key or a continuation of dependencies
                    if line.strip().startswith(':'): # Continuation line
                        dep_val = line.split(':', 1)[1].strip()
                        if dep_val: data['dependencies'].append(dep_val)
                        continue
                    
                    key, val = [x.strip() for x in line.split(':', 1)]
                    k = key.upper().replace(" ", "_")
                    
                    if k == 'DISPLAY_NAME': data['display_name'] = val
                    elif k == 'TYPE': data['service_type'] = val
                    elif k == 'STATE': data['state'] = val
                    elif k == 'WIN32_EXIT_CODE': data['exit_code'] = val
                    elif k == 'BINARY_PATH_NAME': data['binary_path'] = val
                    elif k == 'START_TYPE': data['start_mode'] = val
                    elif k == 'SERVICE_START_NAME': data['start_name'] = val
                    elif k == 'ERROR_CONTROL': data['error_control'] = val
                    elif k == 'LOAD_ORDER_GROUP': data['load_order_group'] = val
                    elif k == 'TAG': data['tag'] = val
                    elif k == 'DEPENDENCIES':
                        if val: data['dependencies'].append(val)
            
            data['dependencies_json'] = json.dumps(data['dependencies'])
            results.append(data)
        return results
    except: return []

def parse_ntp_deep(file_path, host_id):
    """Parses ntp_status.txt with precision and source IP extraction."""
    res = {'host_id': host_id, 'source_file': file_path}
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                if ':' in line:
                    k, v = [x.strip() for x in line.split(':', 1)]
                    key = k.lower().replace(' ', '_')
                    res[key] = v
                    
                    # Extract IP from ReferenceId if present
                    if key == 'referenceid' and 'source IP:' in v:
                        ip_match = re.search(r'source IP:\s+([\d\.]+)', v)
                        if ip_match: res['sync_source_ip'] = ip_match.group(1)
                    
                    # Convert Precision to nanoseconds (approx)
                    if key == 'precision':
                        prec_match = re.search(r'\(([\d\.]+)ns', v)
                        if prec_match: res['precision_ns'] = float(prec_match.group(1))
        return [res]
    except: return []

def parse_open_files_audit(file_path, host_id):
    """Parses open_files.txt capturing user and lock details."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            lines = f.readlines()
        table_active = False
        for line in lines:
            if "Files opened remotely" in line: table_active = True; continue
            if table_active and (line.startswith("---") or not line.strip() or "INFO:" in line): continue
            if table_active:
                # Format: Accessed By, Type, Locks, Path
                parts = re.split(r'\s{2,}', line.strip())
                if len(parts) >= 4:
                    results.append({
                        'host_id': host_id, 'accessed_by': parts[0], 
                        'lock_type': parts[1], 'file_path': parts[3], 
                        'source_file': file_path
                    })
        return results
    except: return []

def check_ics(row):
    n = (str(row.get('service_name', '')) + str(row.get('display_name', ''))).upper()
    return any(k.upper() in n for k in ICS_SERVICE_KEYWORDS)

#### Main Loop

In [ ]:
# --- 1. Initialize Result Lists ---
tasks_all = []
services_raw = []
drivers_all = []
ntp_all = []
open_files_all = []

In [ ]:
# Filter df_file_map for Runtime phase
phase5_targets = df_file_map[df_file_map['data_category'] == 'runtime_and_drivers']


In [ ]:
unique_hosts = phase2_targets['host_id'].unique()


In [ ]:
for h_id in tqdm(unique_hosts, desc="Parsing Runtime State"):
    host_files = phase5_targets[phase5_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        # A. Tasks
        if f_name == 'tasks':
            data = parse_tasklist_enriched(f_path, h_id)
            for item in data: item['hostname'] = h_name
            tasks_all.extend(data)
            
        # B. Services (Consolidating sc query, sc qc, and wmic)
        elif 'services' in f_name or 'service_paths' in f_name:
            if f_path.endswith('.csv'):
                raw_csv = parse_phase5_csv(f_path, h_id)
                data = []
                for r in raw_csv:
                    data.append({
                        'host_id': h_id, 'source_file': f_path, 'hostname': h_name,
                        'service_name': r.get('name'), 'display_name': r.get('displayname'),
                        'state': r.get('state'), 'service_type': r.get('servicetype'),
                        'binary_path': r.get('pathname'), 'start_mode': r.get('startmode'),
                        'start_name': r.get('startname'), 'service_pid': r.get('processid')
                    })
            else:
                data = parse_sc_comprehensive(f_path, h_id)
            
            for item in data: item['hostname'] = h_name
            services_raw.extend(data)
            
        # C. Drivers
        elif 'drivers' in f_name:
            if f_path.endswith('.csv'):
                raw_csv = parse_phase5_csv(f_path, h_id)
                data = []
                for r in raw_csv:
                    # Normalize memory fields from "229,376" to int
                    def to_kb(val): return int(str(val).replace(',', '')) // 1024 if val else 0
                    
                    data.append({
                        'host_id': h_id, 'source_file': f_path, 'hostname': h_name,
                        'module_name': r.get('modulename'), 'display_name': r.get('displayname'),
                        'state': r.get('state'), 'driver_type': r.get('drivertype'),
                        'binary_path': r.get('path'), 'start_mode': r.get('startmode'),
                        'link_date': r.get('linkdate'),
                        'kernel_memory_kb': to_kb(r.get('codebytes')) + to_kb(r.get('pagedpoolbytes')),
                        'is_license_dongle': any(k.upper() in str(r.get('modulename', '')).upper() for k in LICENSE_DONGLE_KEYWORDS)
                    })
            else:
                # drivers_kernel.txt uses sc query format
                data = parse_sc_comprehensive(f_path, h_id)
            
            for item in data: item['hostname'] = h_name
            drivers_all.extend(data)
            
        # D. NTP & Open Files
        elif f_name == 'ntp_status':
            data = parse_ntp_deep(f_path, h_id)
            for item in data: item['hostname'] = h_name
            ntp_all.extend(data)
        elif f_name == 'open_files':
            data = parse_open_files_audit(f_path, h_id)
            for item in data: item['hostname'] = h_name
            open_files_all.extend(data)

In [ ]:
# --- 3. Final Consolidation ---
df_tasks_final = pd.DataFrame(tasks_all)
df_ntp_final = pd.DataFrame(ntp_all)
df_open_files_final = pd.DataFrame(open_files_all)

# Merge services from all sources
df_srv_tmp = pd.DataFrame(services_raw)
df_services_final = df_srv_tmp.groupby(['host_id', 'service_name']).first().reset_index()

# Merge drivers
df_drv_tmp = pd.DataFrame(drivers_all)
# Use module_name as key for drivers
df_drivers_final = df_drv_tmp.groupby(['host_id', 'module_name']).first().reset_index()

# Re-apply ICS flags to merged services

df_services_final['is_ics_related'] = df_services_final.apply(check_ics, axis=1)


### Software Inventory and Compliance
---
**Overview**
This phase reconstructs a 100% complete software inventory. We use `software_inventory.csv` and `software_inventory_x32.csv` as **Schema Templates** to define our target data model. The actual data is extracted by parsing recursive registry dumps (`reg_uninstall_*.txt`) and merging them with MSI-registered application data from WMI. This ensures that both standard installations and "portable" or non-MSI software are captured and normalized.

**Target Artifacts:**
* `software_inventory.csv`: Schema header for 64-bit software (Name, Version, Publisher, InstallDate, UninstallString).
* `software_inventory_x32.csv`: Schema header for 32-bit (WoW64) software.
* `apps_wmic.csv`: MSI-registered applications (Primary source for clean vendor names).
* `reg_uninstall_x64.txt`: Raw recursive registry dump (64-bit hive).
* `reg_uninstall_x32.txt`: Raw recursive registry dump (32-bit hive).
* `patches.csv`: List of installed QFE patches and hotfixes.
* `av_status.csv`: Antivirus product name and protection state.
* `dotnet_versions.txt`: Installed .NET Framework versions.
* `root_certs.txt`: Certificates in the Trusted Root store.

**Output Datasets & Mapping:**

1. **`df_sw_final` (Unified Software Inventory)**
   * *Schema:* Defined by `software_inventory.csv`.
   * *Data Sources:* `reg_uninstall_x64.txt`, `reg_uninstall_x32.txt`, `apps_wmic.csv`.
2. **`df_patches_final` (OS Updates)**
   * *Source:* `patches.csv`.
3. **`df_av_status_final` (Endpoint Protection)**
   * *Source:* `av_status.csv`.
4. **`df_compliance_misc_final` (Frameworks & Certs)**
   * *Sources:* `dotnet_versions.txt`, `root_certs.txt`.


#### Data Schema


| Field | Type | Description |
| :--- | :--- | :--- |
| `host_id` | string | Unique key: `location__system__hostname`. |
| `sw_name` | string | Application name (from DisplayName). |
| `sw_version` | string | Version string (from DisplayVersion). |
| `sw_publisher` | string | Software vendor (from Publisher). |
| `install_date` | string | Date of installation (from InstallDate). |
| `uninstall_string`| string | Command to remove the software (Forensic marker). |
| `architecture` | string | x64 or x32 (based on the source hive). |


#### Local Functions

In [ ]:
def parse_registry_uninstall_recursive(file_path, host_id, arch):
    """
    Parses raw recursive registry dumps from the Uninstall hive.
    Groups all forensic markers (InstallLocation, UninstallString, etc.) 
    under their parent HKEY keys.
    """
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        
        # Split by HKEY headers
        blocks = re.split(r'\n(HKEY_LOCAL_MACHINE\\.*?)\n', content)
        
        for i in range(1, len(blocks), 2):
            key_path = blocks[i].strip()
            properties = blocks[i+1]
            
            app = {
                'host_id': host_id, 'source_file': file_path, 'architecture': arch,
                'registry_key': key_path,
                'sw_name': None, 'sw_version': None, 'sw_publisher': None, 
                'install_date': None, 'install_location': None, 
                'uninstall_string': None, 'is_system_component': False
            }
            
            # Mapping all possible registry values to our fields
            mapping = {
                'DisplayName': 'sw_name',
                'DisplayVersion': 'sw_version',
                'Publisher': 'sw_publisher',
                'InstallDate': 'install_date',
                'InstallLocation': 'install_location',
                'UninstallString': 'uninstall_string'
            }
            
            for reg_name, target in mapping.items():
                # Search for the property line and capture the value after REG_SZ/EXPAND_SZ
                m = re.search(fr'^\s+{reg_name}\s+REG_(?:SZ|EXPAND_SZ)\s+(.*)$', properties, re.MULTILINE)
                if m: app[target] = m.group(1).strip().replace('"', '')

            # Check for SystemComponent flag (REG_DWORD 0x1)
            sys_comp = re.search(r'^\s+SystemComponent\s+REG_DWORD\s+0x1', properties, re.MULTILINE)
            if sys_comp: app['is_system_component'] = True
            
            if app['sw_name']:
                # Normalize InstallDate (YYYYMMDD -> datetime)
                if app['install_date'] and len(str(app['install_date'])) == 8:
                    try: app['install_date'] = pd.to_datetime(app['install_date'], format='%Y%m%d')
                    except: pass
                results.append(app)
        return results
    except: return []


def parse_root_certs(file_path, host_id):
    """Parses certutil -store root output into a structured list."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        
        blocks = content.split('================ Certificate')
        for block in blocks[1:]:
            cert = {'host_id': host_id, 'source_file': file_path, 'type': 'Root Certificate'}
            
            # Extract fields using anchors
            fields = {
                'Serial Number': 'serial_number',
                'Issuer': 'issuer',
                'NotBefore': 'valid_from',
                'NotAfter': 'valid_to',
                'Subject': 'subject',
                'Cert Hash(sha1)': 'thumbprint_sha1'
            }
            
            for anchor, field in fields.items():
                match = re.search(fr'{anchor}:\s+(.*)', block)
                if match: cert[field] = match.group(1).strip()
            
            results.append(cert)
        return results
    except: return []

def parse_dotnet_registry(file_path, host_id):
    """Extracts .NET versions from registry dump with full path context."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        # Find all Version REG_SZ lines and their parent keys
        matches = re.findall(r'(HKEY_LOCAL_MACHINE\\.*?)\n.*?Version\s+REG_SZ\s+([\d\.]+)', content, re.DOTALL)
        for key, ver in matches:
            results.append({
                'host_id': host_id, 'source_file': file_path,
                'component': 'NetFramework', 'sub_component': key.split('\\')[-1],
                'version': ver
            })
        return results
    except: return []
        
def _standardize_patch_date(raw_val):
    """
    Ultimate forensic date normalizer with precision truncation.
    Normalizes all dates to midnight (00:00:00).
    """
    s = str(raw_val).replace('\x00', '').strip().lower()
    s = re.sub(r'[^a-f0-9/.\-]', '', s)
    
    if not s or s in ['nan', '0', 'none']:
        return pd.NaT
    
    # 1. Handle Windows FileTime Hex
    if len(s) == 16 and s.startswith('01'):
        try:
            filetime_int = int(s, 16)
            unix_offset = 11644473600
            unix_ts = (filetime_int / 10000000) - unix_offset
            # Convert and NORMALIZE to midnight
            return pd.to_datetime(unix_ts, unit='s').normalize()
        except: 
            pass

    # 2. Handle standard date strings
    for day_first in [False, True]:
        try:
            d = pd.to_datetime(s, dayfirst=day_first, errors='raise')
            if 1990 < d.year < 2030:
                # NORMALIZE to midnight
                return d.normalize()
        except: 
            continue
            
    return pd.NaT



#### Main Loop

In [ ]:
# --- 1. Initialize Result Lists ---
sw_registry_list = []
sw_wmi_list = []
patches_all = []
av_status_all = []
compliance_misc = []

In [ ]:
phase6_targets = df_file_map[df_file_map['data_category'] == 'software_and_compliance']

In [ ]:
unique_hosts = phase6_targets['host_id'].unique()


In [ ]:
for h_id in tqdm(unique_hosts, desc="Auditing Software"):
    host_files = phase6_targets[phase6_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        # 1. Registry Software (Primary Source for Metadata)
        if 'reg_uninstall' in f_name:
            arch = 'x64' if 'x64' in f_name else 'x86_wow64'
            data = parse_registry_uninstall_recursive(f_path, h_id, arch)
            for item in data: item['hostname'] = h_name
            sw_registry_list.extend(data)
            
        # 2. WMI Software (Supplementary Source)
        elif 'apps_wmic' in f_name:
            data, status = parse_disks(f_path, h_id)
            if "SUCCESS" in status:
                for item in data:
                    sw_wmi_list.append({
                        'host_id': h_id, 'hostname': h_name, 'sw_name': item.get('Name'), 
                        'sw_version': item.get('Version'), 'sw_publisher': item.get('Vendor'),
                        'architecture': 'msi_registered'
                    })
                    
        # 3. Patches, AV, .NET, Certs (Standard logic)
        elif 'patches' in f_name:
            data, status = parse_disks(f_path, h_id)
            if "SUCCESS" in status:
                for item in data:
                    patches_all.append({
                        'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                        'kb_id': item.get('HotFixID'), 'description': item.get('Description'),
                        'installed_by': item.get('InstalledBy'), 
                        'installed_on': _standardize_patch_date(item.get('InstalledOn'))
                    })
        elif 'av_status' in f_name:
            data, status = parse_disks(f_path, h_id)
            if "SUCCESS" in status:
                for item in data:
                    av_status_all.append({
                        'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                        'av_product_name': item.get('displayName'), 'av_product_state': item.get('productState')
                    })
        elif 'dotnet_versions' in f_name:
            data = parse_dotnet_registry(f_path, h_id)
            for item in data: item['hostname'] = h_name
            compliance_misc.extend(data)
        elif 'root_certs' in f_name:
            data = parse_root_certs(f_path, h_id)
            for item in data: item['hostname'] = h_name
            compliance_misc.extend(data)

In [ ]:
df_reg = pd.DataFrame(sw_registry_list)
df_wmi = pd.DataFrame(sw_wmi_list)

# Merge: Registry is the base, WMI fills missing publishers/apps
df_sw_final = pd.merge(
    df_reg, df_wmi, 
    on=['host_id', 'sw_name', 'sw_version'], 
    how='outer', suffixes=('', '_wmi')
)

df_sw_final['sw_publisher'] = df_sw_final['sw_publisher'].fillna(df_sw_final['sw_publisher_wmi'])
df_sw_final['hostname'] = df_sw_final['hostname'].fillna(df_sw_final['hostname_wmi'])
df_sw_final['architecture'] = df_sw_final['architecture'].fillna('msi_registered')
df_sw_final = df_sw_final.drop(columns=['sw_publisher_wmi', 'hostname_wmi'])

df_patches_final = pd.DataFrame(patches_all).dropna(subset=['kb_id'])
df_av_status_final = pd.DataFrame(av_status_all)
df_compliance_misc_final = pd.DataFrame(compliance_misc)


### Persistence Mechanisms and Group Policy
---
**Overview**
This phase audits the mechanisms used by the system to maintain state across reboots and the governance framework enforced by Group Policy Objects (GPO). In ICS environments, persistence is often achieved through scheduled maintenance tasks or vendor-specific autorun keys. Auditing these, alongside the local security policy (Secedit) and firewall rules, is critical for detecting unauthorized changes, "living-off-the-land" techniques, and ensuring compliance with the site's security baseline.

**Target Artifacts:**
* `scheduled_tasks.csv`: Comprehensive list of all tasks (Name, Trigger, Action, User context).
* `run_hklm.txt` / `run_hkcu.txt`: Standard registry autorun keys for system and user contexts.
* `runonce_hklm.txt` / `runonce_hkcu.txt`: One-time startup commands.
* `ifeo.txt`: Registry dump of Image File Execution Options (used for debugger settings or hijacks).
* `local_policies.inf`: Export of the local security policy (User Rights, Audit, Account config).
* `firewall_rules.txt`: Configuration and active rules of the Windows Firewall (netsh dump).
* `GPO_Raw` (Folder): Raw GPO files including `registry.pol` and `GptTmpl.inf`.

**Output Datasets & Mapping:**

1. **`df_tasks_scheduled_final` (Scheduled Tasks)**
   * *Source:* `scheduled_tasks.csv`.
2. **`df_autoruns_final` (Registry Persistence)**
   * *Sources:* `run_*.txt`, `runonce_*.txt`, `ifeo.txt`.
3. **`df_security_policies_final` (Local Security Policy)**
   * *Source:* `local_policies.inf`.
4. **`df_firewall_rules_final` (Host Firewall)**
   * *Source:* `firewall_rules.txt`.

**Data Schema:**

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_tasks_scheduled_final** | `task_name` | string | Full path and name of the task. |
| | `task_to_run` | string | The actual command or script being executed. |
| | `run_as_user` | string | The account context used for execution. |
| | `scheduled_days` | string | Frequency and trigger details. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_autoruns_final** | `reg_key` | string | The specific registry key path. |
| | `entry_name` | string | The name of the autorun entry. |
| | `value_data` | string | The command line or path to the executable. |
| | `hive` | string | HKLM or HKCU. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_security_policies_final** | `section` | string | Policy category (e.g., Privilege Rights, System Access). |
| | `policy_name` | string | Technical name of the setting. |
| | `policy_value` | string | Configured value or assigned SID/User. |
| | `parameter_type` | string | Human-readable classification (e.g., Audit Policy). |


#### Local Functions

In [ ]:
import pandas as pd
import re
import os
import json
import io

def get_gpo_extension_name(guid):
    """Maps GPO Extension GUIDs to human-readable names."""
    guid_map = {
        '{35378EAC-683F-11D2-A89A-00C04FBBCFA2}': 'Registry Settings',
        '{8FC0B734-A0E1-11D1-A7D3-0000F87571E3}': 'Security Settings',
        '{D02B1F72-3407-48AE-BA88-E8213C6761F1}': 'Scripts (Startup/Shutdown)',
        '{D02B1F73-3407-48AE-BA88-E8213C6761F1}': 'Scripts (Logon/Logoff)',
        '{42B5FA30-D6EE-11D2-BB05-00C04F7956E4}': 'IP Security',
        '{B352D13D-24AD-11D1-A147-00A0C9062910}': 'Software Installation',
        '{C6DC5466-781F-4C8E-8B2C-9609930BC770}': 'Software Installation (User)',
        '{A2E30F80-D7A0-11D2-BA21-0000F87571E3}': 'Internet Explorer Branding',
        '{A2E30F80-D7DE-11D2-BBDE-00C04F86AE3B}': 'Internet Explorer Maintenance',
        '{00000000-0000-0000-0000-000000000000}': 'Core GPO Engine'
    }
    return guid_map.get(guid.upper(), f"Unknown Extension ({guid})")

def parse_security_policies(file_path, host_id):
    """Parses policies.inf and GptTmpl.inf with 100% coverage and privilege explosion."""
    policy_rows = []
    current_section = None
    type_map = {
        'SYSTEM ACCESS': 'Account Policy',
        'EVENT AUDIT': 'Audit Policy',
        'PRIVILEGE RIGHTS': 'User Right Assignment',
        'REGISTRY VALUES': 'Registry Security',
        'KERBEROS POLICY': 'Kerberos Policy',
        'VERSION': 'Metadata',
        'UNICODE': 'Metadata'
    }
    try:
        content = None
        for enc in ['utf-16', 'utf-8', 'cp1251']:
            try:
                with open(file_path, 'r', encoding=enc) as f:
                    test_content = f.read()
                if '[' in test_content and ']' in test_content:
                    content = test_content
                    break
            except: continue
        if not content: return [], "FAILED: Encoding issue"

        for line in content.splitlines():
            line = line.strip()
            if not line: continue
            if line.startswith('[') and line.endswith(']'):
                current_section = line.strip('[]').upper()
                continue
            if '=' in line and current_section:
                key, raw_val = [x.strip() for x in line.split('=', 1)]
                raw_val = raw_val.replace('"', '')
                param_type = type_map.get(current_section, 'Other Configuration')
                
                if current_section == 'PRIVILEGE RIGHTS':
                    for val in [v.strip() for v in raw_val.split(',') if v.strip()]:
                        policy_rows.append({'host_id': host_id, 'parameter_type': param_type, 'section': current_section, 'policy_name': key, 'policy_value': val, 'source_file': file_path})
                elif current_section == 'REGISTRY VALUES':
                    reg_parts = raw_val.split(',', 1)
                    policy_rows.append({'host_id': host_id, 'parameter_type': param_type, 'section': current_section, 'policy_name': key, 'policy_value': reg_parts[1] if len(reg_parts)>1 else raw_val, 'reg_type': reg_parts[0] if len(reg_parts)>1 else None, 'source_file': file_path})
                else:
                    policy_rows.append({'host_id': host_id, 'parameter_type': param_type, 'section': current_section, 'policy_name': key, 'policy_value': raw_val, 'source_file': file_path})
        return policy_rows, f"SUCCESS: {len(policy_rows)} entries"
    except Exception as e: return [], f"ERROR: {str(e)}"

def parse_registry_pol(file_path, host_id, integrity_status):
    """Binary parser for Registry Policy (PReg) files."""
    reg_entries = []
    try:
        if os.path.getsize(file_path) < 8: return [], "FAILED: Too small"
        with open(file_path, 'rb') as f:
            header = f.read(8)
            if header[:4] != b'PReg': return [], "FAILED: Not PReg"
            raw_data = f.read().decode('utf-16-le', errors='ignore')
        blocks = re.findall(r'\[(.*?)]', raw_data, re.DOTALL)
        for block in blocks:
            parts = block.split(';')
            if len(parts) < 5: continue
            value_name = parts[1].strip()
            is_del = value_name.startswith("**del.")
            reg_entries.append({
                'host_id': host_id, 'reg_key': parts[0].strip(),
                'reg_value': value_name.replace("**del.", "") if is_del else value_name,
                'reg_data': parts[4].strip(), 'reg_type_hint': parts[2].strip(),
                'is_deletion': is_del, 'integrity_status': integrity_status, 'source_file': file_path
            })
        return reg_entries, f"SUCCESS: {len(reg_entries)} entries"
    except Exception as e: return [], f"ERROR: {str(e)}"

def parse_gpo_automation(file_path, host_id):
    """Parses gpt.ini (metadata) and scripts.ini (automation)."""
    results = []
    file_name = os.path.basename(file_path).lower()
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        if file_name == 'gpt.ini':
            v_match = re.search(r'Version=(\d+)', content)
            v_raw = int(v_match.group(1)) if v_match else 0
            guids = re.findall(r'\{[A-F0-9-]+\}', content, re.IGNORECASE)
            results.append({
                'host_id': host_id, 'gpo_component': 'Metadata', 'script_type': 'General',
                'machine_revision': v_raw & 0xFFFF, 'user_revision': (v_raw >> 16) & 0xFFFF,
                'active_extensions': json.dumps(list(set([get_gpo_extension_name(g) for g in guids]))),
                'source_file': file_path
            })
        elif file_name == 'scripts.ini':
            context = 'SYSTEM' if 'Machine' in file_path else 'User'
            sections = re.findall(r'\[(.*?)\](.*?)(?=\[|$)', content, re.DOTALL)
            for sec_name, sec_content in sections:
                cmds = re.findall(r'(\d+)CmdLine=(.*)', sec_content)
                params = dict(re.findall(r'(\d+)Parameters=(.*)', sec_content))
                for idx, cmd_path in cmds:
                    results.append({
                        'host_id': host_id, 'gpo_component': 'Automation', 'script_type': sec_name.strip(),
                        'command_line': cmd_path.strip(), 'parameters': params.get(idx, "").strip(),
                        'execution_context': context, 'source_file': file_path
                    })
        return results, f"SUCCESS: {len(results)} entries"
    except Exception as e: return [], f"ERROR: {str(e)}"

def parse_scheduled_tasks_csv(file_path, host_id):
    """Parses scheduled_tasks.csv with full metadata."""
    try:
        with open(file_path, 'rb') as f:
            raw = f.read().replace(b'\x00', b'')
        df_tmp = pd.read_csv(io.StringIO(raw.decode('cp1251', errors='ignore')), quotechar='"', on_bad_lines='skip', engine='python')
        df_tmp.columns = [c.strip().replace(' ', '_').lower() for c in df_tmp.columns]
        res = df_tmp.to_dict('records')
        for item in res: item['host_id'] = host_id; item['source_file'] = file_path
        return res
    except: return []

def parse_registry_run_keys(file_path, host_id):
    """Parses raw 'reg query' output for Run, RunOnce, and IFEO keys (recursive)."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        blocks = re.split(r'\n(HKEY_.*?)\n', content)
        for i in range(1, len(blocks), 2):
            key_path = blocks[i].strip()
            properties = blocks[i+1]
            matches = re.findall(r'^\s+(.*?)\s+(REG_(?:SZ|EXPAND_SZ|DWORD|QWORD|BINARY))\s+(.*)$', properties, re.MULTILINE)
            for m in matches:
                results.append({
                    'host_id': host_id, 'registry_key': key_path, 'entry_name': m[0].strip(),
                    'value_type': m[1].strip(), 'value_data': m[2].strip(),
                    'hive': 'HKLM' if 'LOCAL_MACHINE' in key_path else 'HKCU', 'source_file': file_path
                })
        return results
    except: return []

def parse_firewall_rules(file_path, host_id):
    """Parses netsh advfirewall dump (Profiles and Rules)."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        blocks = content.split('Rule Name:')
        for block in blocks[1:]:
            lines = block.splitlines()
            rule = {'host_id': host_id, 'rule_name': lines[0].strip(), 'source_file': file_path}
            for line in lines[1:]:
                if ':' in line:
                    k, v = [x.strip() for x in line.split(':', 1)]
                    rule[k.lower().replace(' ', '_')] = v
            results.append(rule)
        return results
    except: return []

#### Main loop

In [ ]:
tasks_scheduled_list = []
autoruns_list = []
security_policies_list = []
firewall_rules_list = []

In [ ]:
phase7_targets = df_file_map[df_file_map['data_category'] == 'persistence_and_gpo']
unique_hosts = phase7_targets['host_id'].unique()

In [ ]:
phase7_targets.info()

In [ ]:
for h_id in tqdm(unique_hosts, desc="Parsing Persistence & GPO"):
    host_files = phase7_targets[phase7_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        # 1. Scheduled Tasks (CSV)
        if 'scheduled_tasks' in f_name:
            data = parse_scheduled_tasks_csv(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                tasks_scheduled_list.append(item)
        
        # 2. Registry Autoruns and IFEO (Recursive Reg Dumps)
        elif any(x in f_name for x in ['run_hklm', 'run_hkcu', 'runonce', 'ifeo']):
            data = parse_registry_run_keys(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                autoruns_list.append(item)
        
        # 3. Local Security Policies (Secedit / INF)
        elif 'local_policies' in f_name:
            data, status = parse_security_policies(f_path, h_id)
            if data:
                for item in data:
                    item['hostname'] = h_name
                    security_policies_list.append(item)
        
        # 4. Firewall Rules (Netsh dump)
        elif 'firewall_rules' in f_name:
            data = parse_firewall_rules(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                firewall_rules_list.append(item)
        
        # 5. Raw GPO Files (Registry.pol, gpt.ini, scripts.ini)
        # These are handled by specialized GPO parsers if they exist in GPO_Raw
        elif f_name == 'gpt' or f_name == 'scripts':
            data, status = parse_gpo_automation(f_path, h_id)
            if data:
                # These results are usually merged into a separate GPO metadata table
                # but for now, we ensure they are captured
                for item in data:
                    item['hostname'] = h_name
                    # You can choose to add these to a specific GPO dataframe
                    # For this loop, we'll focus on the 4 primary datasets
                    pass

In [ ]:
df_tasks_scheduled_final = pd.DataFrame(tasks_scheduled_list)
df_autoruns_final = pd.DataFrame(autoruns_list)
df_security_policies_final = pd.DataFrame(security_policies_list)
df_firewall_rules_final = pd.DataFrame(firewall_rules_list)


### OT-Specific Hardware Scan
---
**Overview**
This phase identifies specialized industrial hardware and communication interfaces that are often critical for ICS operations but invisible to standard IT asset management tools. By auditing Plug-and-Play (PnP) devices, serial (COM) ports, and USB controllers, we map the physical connectivity of the host to the plant floor. This data is essential for identifying "Single Points of Failure" (e.g., a unique Profibus card) and detecting unauthorized hardware attachments.

**Target Artifacts:**
* `pnp.csv`: Full enumeration of all PnP devices (Name, Device ID, Status, Class).
* `com_ports.csv`: Serial port configurations (Name, Port, Baud rate, Parity).
* `usb_ctrl.csv`: USB controllers and their associated device references.
* `parallel_ports.csv`: Legacy LPT ports, still common in older industrial setups.
* `sound_devices.csv`: Audio hardware, sometimes used for specific ICS signaling or alerts.

**Output Datasets & Mapping:**

1. **`df_pnp_devices_final` (PnP Inventory)**
   * *Source:* `pnp.csv`.
2. **`df_com_ports_final` (Serial Connectivity)**
   * *Source:* `com_ports.csv`.
3. **`df_usb_controllers_final` (USB Topology)**
   * *Source:* `usb_ctrl.csv`.
4. **`df_legacy_ports_final` (LPT & Sound)**
   * *Sources:* `parallel_ports.csv`, `sound_devices.csv`.



#### Data Schema


| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_pnp_devices_final** | `name` | string | Friendly name of the device. |
| | `deviceid` | string | Unique hardware instance ID (contains Vendor/Product IDs). |
| | `status` | string | Operational state (OK, Error, Degraded). |
| | `service` | string | Driver service associated with the device. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_com_ports_final** | `name` | string | COM port name (e.g., Communications Port (COM1)). |
| | `deviceid` | string | System identifier for the serial interface. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_usb_controllers_final** | `dependent` | string | Reference to the device connected to the controller. |
| | `antecedent` | string | Reference to the USB controller itself. |


#### Local Functions

In [ ]:
import csv
import io
import re

def parse_ot_hardware_csv(file_path, host_id):
    """
    Ultimate WMIC CSV parser. 
    Uses python's csv module to handle nested quotes and commas correctly.
    """
    results = []
    try:
        # 1. Read raw bytes and remove nulls
        with open(file_path, 'rb') as f:
            raw_bin = f.read().replace(b'\x00', b'')
        
        if not raw_bin: return []

        # 2. Decode with BOM handling
        content = None
        for enc in ['utf-16', 'utf-8', 'cp1251']:
            try:
                content = raw_bin.decode(enc, errors='ignore')
                if 'Node' in content: break
            except: continue
        
        if not content: return []

        # 3. Use csv.DictReader with flexible settings
        # We strip the content to remove leading/trailing empty lines
        f_io = io.StringIO(content.strip())
        
        # We use a normal reader first to clean the headers manually
        reader = csv.reader(f_io)
        raw_headers = next(reader, None)
        if not raw_headers: return []
        
        # Clean headers: remove BOM, quotes, spaces and lowercase them
        headers = [re.sub(r'[^\w]', '', h).lower() for h in raw_headers]
        
        # 4. Process data rows
        for row in reader:
            if not row: continue
            # Map cleaned headers to row values
            item = {}
            for i, h in enumerate(headers):
                if i < len(row):
                    val = row[i].strip().strip('"').replace('&amp;', '&')
                    # Convert empty strings or 'none' to None object
                    item[h] = val if val and val.lower() != 'none' else None
                else:
                    item[h] = None
            
            item['host_id'] = host_id
            item['source_file'] = file_path
            results.append(item)
            
        return results
    except:
        return []

def extract_wmi_device_id(wmi_path):
    """
    Extracts DeviceID from complex WMI paths like \\HOST\root...DeviceID="ID"
    """
    if not wmi_path: return None
    s = str(wmi_path).replace('&amp;', '&')
    # Try to find content inside double quotes
    match = re.search(r'="(.+?)"', s)
    if match: return match.group(1)
    # Fallback: take everything after the last '='
    if '=' in s: return s.split('=')[-1].strip('"')
    return s.strip('"')

Main Loop

In [ ]:
#  OT-Specific Hardware Scan Execution Loop ---
pnp_results = []
com_results = []
usb_results = []
legacy_hw_results = []

In [ ]:
phase8_targets = df_file_map[df_file_map['data_category'] == 'ot_hardware']
unique_hosts = phase8_targets['host_id'].unique()

In [ ]:
for h_id in tqdm(unique_hosts, desc="Parsing OT Hardware"):
    host_files = phase8_targets[phase8_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        data = parse_ot_hardware_csv(f_path, h_id)
        if not data: continue
        
        for item in data:
            item['hostname'] = h_name
            
            # 1. PnP Devices
            if f_name == 'pnp':
                pnp_results.append({
                    'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                    'pnp_name': item.get('name') or item.get('caption') or item.get('description'),
                    'pnp_class': item.get('pnpclass'),
                    'device_id': item.get('deviceid') or item.get('pnpdeviceid'),
                    'service': item.get('service'),
                    'status': item.get('status'),
                    'manufacturer': item.get('manufacturer')
                })
            
            # 2. COM Ports
            elif f_name == 'com_ports':
                com_results.append({
                    'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                    'port_name': item.get('name') or item.get('caption'),
                    'device_id': item.get('deviceid'),
                    'status': item.get('status')
                })
                
            # 3. USB Topology
            elif f_name == 'usb_ctrl':
                usb_results.append({
                    'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                    'controller_id': extract_wmi_device_id(item.get('antecedent')),
                    'device_id': extract_wmi_device_id(item.get('dependent'))
                })
                
            # 4. Legacy Ports & Sound
            elif f_name in ['parallel_ports', 'sound_devices']:
                legacy_hw_results.append({
                    'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                    'hw_type': 'Parallel Port' if 'parallel' in f_name else 'Sound Device',
                    'name': item.get('name') or item.get('caption'),
                    'device_id': item.get('deviceid'),
                    'status': item.get('status')
                })

In [ ]:
# --- Create Final OT Hardware Datasets ---
df_pnp_devices_final = pd.DataFrame(pnp_results)
df_com_ports_final = pd.DataFrame(com_results)
df_usb_topology_final = pd.DataFrame(usb_results)
df_legacy_hardware_final = pd.DataFrame(legacy_hw_results)

### Forensic Event Logs
---
**Overview**
This phase focuses on the system's forensic trail. We audit the **USB History** by parsing the `USBSTOR` registry hive, which reveals every external storage device ever connected to the host—a critical marker for "Air-gap" security analysis. Additionally, we inventory all available **Event Log Channels** and preserve the primary binary logs (`Security`, `System`, `Application`) for external deep-dive forensic analysis.

**Target Artifacts:**
* `event_logs_list.txt`: Complete list of registered event log channels.
* `usb_history.txt`: Recursive registry dump of `HKLM\SYSTEM\CurrentControlSet\Enum\USBSTOR`.
* `Security.evtx` / `System.evtx` / `Application.evtx`: Binary event log exports.

**Output Datasets & Mapping:**

1. **`df_usb_history_final` (USB Forensic Registry)**
   * *Source:* `usb_history.txt`.
2. **`df_event_logs_inventory_final` (Log Channel Audit)**
   * *Source:* `event_logs_list.txt`.
3. **`df_evtx_metadata_final` (Binary Log Registry)**
   * *Sources:* `*.evtx`.


#### Data Schemas


| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_usb_history_final** | `host_id` | string | Unique key: `location__system__hostname`. |
| | `device_instance_id` | string | Unique ID (contains Vendor, Product, and Serial). |
| | `friendly_name` | string | Human-readable device name (e.g., "Kingston DataTraveler"). |
| | `device_description` | string | System description of the hardware class. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_event_logs_inventory_final** | `channel_name` | string | Name of the registered log channel. |

| Dataset | Field | Type | Description |
| :--- | :--- | :--- | :--- |
| **df_evtx_metadata_final** | `log_name` | string | Name of the binary log (Security, System, etc.). |
| | `file_size_mb` | float | Size of the log file (indicates log retention depth). |


#### Local Functions

In [ ]:

def parse_usb_history(file_path, host_id):
    raw_entries = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        blocks = re.split(r'\n(HKEY_LOCAL_MACHINE\\.*?)\n', content)
        for i in range(1, len(blocks), 2):
            key_path = blocks[i].strip()
            properties = blocks[i+1]
            parts = key_path.split('\\')
            if len(parts) >= 7:
                entry = {
                    'device_id': parts[5], 'serial_number': parts[6],
                    'friendly_name': None, 'device_desc': None, 'manufacturer': None
                }
                fn = re.search(r'^\s+FriendlyName\s+REG_SZ\s+(.*)$', properties, re.MULTILINE)
                if fn: entry['friendly_name'] = fn.group(1).strip()
                dd = re.search(r'^\s+DeviceDesc\s+REG_SZ\s+(.*)$', properties, re.MULTILINE)
                if dd: entry['device_desc'] = dd.group(1).strip().split(';')[-1]
                mfg = re.search(r'^\s+Mfg\s+REG_SZ\s+(.*)$', properties, re.MULTILINE)
                if mfg: entry['manufacturer'] = mfg.group(1).strip().split(';')[-1]
                raw_entries.append(entry)
        if not raw_entries: return []
        df_temp = pd.DataFrame(raw_entries).replace(['None', ''], [None, None])
        df_agg = df_temp.groupby(['device_id', 'serial_number']).first().reset_index()
        res = df_agg.to_dict('records')
        for item in res:
            item['host_id'] = host_id
            item['source_file'] = file_path
        return res
    except: return []

def parse_event_channels(file_path, host_id):
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            for line in f:
                channel = line.strip()
                if channel:
                    results.append({'host_id': host_id, 'source_file': file_path, 'channel_name': channel})
        return results
    except: return []

def parse_evtx_security_critical(file_path, host_id, pbar=None):
    """
    High-performance EVTX parser. 
    Uses Regex for fast ID filtering and captures all Data fields.
    """
    try:
        import Evtx.Evtx as evtx
    except ImportError: return []

    results = []
    # Expanded list of critical IDs for better visibility
    # 4624: Logon, 4625: Failed, 4672: Admin Logon, 1102: Audit Clear, 4720: User Created
    TARGET_IDS = {'4624', '4625', '4672', '1102', '4720'}
    
    record_count = 0
    match_count = 0
    
    try:
        with evtx.Evtx(file_path) as log:
            for record in log.records():
                record_count += 1
                xml_str = record.xml()
                
                # 1. Fast ID detection using Regex (handles attributes like Qualifiers=...)
                id_match = re.search(r'<EventID.*?>(?P<eid>\d+)</EventID>', xml_str)
                if id_match:
                    eid = id_match.group('eid')
                    
                    if eid in TARGET_IDS:
                        match_count += 1
                        # 2. Parse XML only for matched records
                        root = ET.fromstring(xml_str)
                        ns = {'ns': 'http://schemas.microsoft.com/win/2004/08/events/event'}
                        
                        ts = root.find('.//ns:TimeCreated', ns).get('SystemTime') if root.find('.//ns:TimeCreated', ns) is not None else None
                        
                        # 3. Greedy Data Extraction: Capture all <Data> fields
                        event_data = {}
                        for data in root.findall('.//ns:Data', ns):
                            name = data.get('Name')
                            if name:
                                event_data[name] = data.text
                        
                        results.append({
                            'host_id': host_id, 
                            'event_id': eid, 
                            'timestamp': ts, 
                            'user': event_data.get('TargetUserName') or event_data.get('SubjectUserName') or "N/A",
                            'ip_address': event_data.get('IpAddress') or "N/A",
                            'all_data_json': json.dumps(event_data),
                            'log_name': os.path.basename(file_path),
                            'source_file': file_path
                        })

                # Update progress bar every 2000 records
                if pbar and record_count % 2000 == 0:
                    pbar.set_postfix(status=f"Found:{match_count}", total_recs=record_count)
                    
        return results
    except Exception as e:
        print(f" [!] Error in {file_path}: {e}")
        return []

# --- Keep existing parse_usb_history and parse_event_channels from previous turn ---

#### Main Loop

In [ ]:
usb_history_list = []
event_channels_list = []
event_logs_critical_list = []


In [ ]:
# Filter targets for Phase 9
phase9_targets = df_file_map[df_file_map['data_category'] == 'event_logs']
unique_hosts = phase9_targets['host_id'].unique()


In [ ]:
pbar = tqdm(unique_hosts, desc="Forensic Ingestion")

for h_id in pbar:
    host_files = phase9_targets[phase9_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    pbar.set_description(f"Forensic Ingestion: {h_name}")
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        f_ext = row['extension'].lower()
        
        if 'usb_history' in f_name:
            data = parse_usb_history(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                usb_history_list.append(item)
        
        elif 'event_logs_list' in f_name:
            data = parse_event_channels(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                event_channels_list.append(item)
        
        elif f_ext == '.evtx' and 'security' in f_name:
            # This will now correctly find events and update the postfix
            data = parse_evtx_security_critical(f_path, h_id, pbar=pbar)
            for item in data:
                item['hostname'] = h_name
                event_logs_critical_list.append(item)

In [ ]:
# --- Create Final Datasets ---
df_usb_history_final = pd.DataFrame(usb_history_list)
df_event_channels_final = pd.DataFrame(event_channels_list)
df_event_logs_critical_final = pd.DataFrame(event_logs_critical_list)

### Volatile Forensic Snapshot
---
**Overview**
This phase reconstructs the system's execution history and user activity by auditing volatile forensic artifacts. These "traces of evidence" allow us to see what programs were run, when they were executed, and by whom, even if the original files have been deleted. By correlating **UserAssist**, **Prefetch**, **BAM**, and **ShimCache**, we build a high-resolution timeline of system usage, which is critical for detecting unauthorized software and "Living-off-the-Land" (LotL) attacks.

**Target Artifacts:**
* `userassist.txt`: GUI program execution history (ROT13 encoded in the registry).
* `prefetch_list.txt`: List of `.pf` files with their last execution timestamps.
* `bam_activity.txt`: Background Activity Moderator (BAM) execution timestamps (Windows 10+).
* `muicache.txt`: Friendly names and publisher info for executed programs.
* `shimcache.txt`: Application Compatibility Cache (binary execution history).
* `process_paths.csv`: Full executable paths for all currently running processes.
* `recent_files.txt`: Recently accessed files and documents per user.
* `temp_contents.txt`: Inventory of files in TEMP directories (common malware staging zones).

**Output Datasets & Mapping:**

1. **`df_volatile_execution_final` (The Execution Timeline)**
   * *Sources:* `userassist.txt`, `bam_activity.txt`, `prefetch_list.txt`, `shimcache.txt`.
2. **`df_volatile_muicache_final` (Application Metadata)**
   * *Source:* `muicache.txt`.
3. **`df_volatile_files_final` (Recent & Temp Inventory)**
   * *Sources:* `recent_files.txt`, `temp_contents.txt`.
4. **`df_process_paths_final` (Process-to-File Mapping)**
   * *Source:* `process_paths.csv`.


#### Data Schema


| Field | Type | Description |
| :--- | :--- | :--- |
| `host_id` | string | Unique key: `location__system__hostname`. |
| `evidence_source` | string | Artifact type (UserAssist, BAM, Prefetch, etc.). |
| `executable_path` | string | Full path or name of the executed program. |
| `execution_timestamp`| datetime | Last known run time (normalized). |
| `user_sid` | string | SID of the user associated with the activity (for BAM/UserAssist). |


#### Local Functions

In [ ]:

def decode_filetime(hex_str):
    """Decodes 64-bit Little-Endian Hex (Windows FileTime) to datetime."""
    try:
        s = re.sub(r'[^0-9A-F]', '', str(hex_str).upper())
        if len(s) < 16: return pd.NaT
        binary_data = binascii.unhexlify(s[:16])
        filetime = struct.unpack('<Q', binary_data)[0]
        if filetime == 0: return pd.NaT
        return datetime(1601, 1, 1) + timedelta(microseconds=filetime // 10)
    except: return pd.NaT

def decode_rot13(text):
    """Decodes ROT13 encoded strings used in UserAssist."""
    if not text: return text
    abc = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    rt13 = "nopqrstuvwxyzabcdefghijklmNOPQRSTUVWXYZABCDEFGHIJKLM"
    table = str.maketrans(abc, rt13)
    return text.translate(table)

def parse_userassist_dump(file_path, host_id):
    """Parses UserAssist registry dumps with ROT13 and timestamp decoding."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        matches = re.findall(r'^\s+(.*?)\s+REG_BINARY\s+([0-9A-F]+)', content, re.MULTILINE)
        for name_rot, hex_data in matches:
            name_clean = decode_rot13(name_rot.strip())
            if name_clean.startswith('UEME_'): continue
            ts = decode_filetime(hex_data[120:136]) if len(hex_data) >= 136 else pd.NaT
            results.append({
                'host_id': host_id, 'source_file': file_path, 'evidence_source': 'UserAssist',
                'executable_path': name_clean, 'execution_timestamp': ts
            })
        return results
    except: return []

def parse_bam_dump(file_path, host_id):
    """Parses BAM registry dumps with binary timestamp decoding."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        blocks = re.split(r'UserSettings\\(S-1-5-[\d-]+)', content)
        for i in range(1, len(blocks), 2):
            sid, props = blocks[i], blocks[i+1]
            matches = re.findall(r'^\s+(\\Device\\.*?)\s+REG_BINARY\s+([0-9A-F]+)', props, re.MULTILINE)
            for path, hex_val in matches:
                results.append({
                    'host_id': host_id, 'source_file': file_path, 'evidence_source': 'BAM',
                    'user_sid': sid, 'executable_path': path, 
                    'execution_timestamp': decode_filetime(hex_val)
                })
        return results
    except: return []

def parse_volatile_list(file_path, host_id, mode='prefetch'):
    """
    Hybrid parser for 'dir' output. 
    Handles both standard tables (with dates) and bare paths (dir /b).
    """
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            for line in f:
                clean_line = line.strip()
                if not clean_line or "Directory of" in clean_line or "Volume in drive" in clean_line:
                    continue
                
                # --- Logic A: Bare Paths (e.g., from temp_contents.txt) ---
                # Matches lines starting with a drive letter like C:\
                if re.match(r'^[a-zA-Z]:\\', clean_line):
                    results.append({
                        'host_id': host_id, 'source_file': file_path, 'evidence_source': mode.upper(),
                        'execution_timestamp': pd.NaT, # No time in bare format
                        'executable_path': clean_line
                    })
                    continue

                # --- Logic B: Standard DIR Table (e.g., from recent_files.txt) ---
                # Regex for: 02/20/2025  11:13 PM  63,927 filename.exe
                match = re.search(r'^(\d{2}[./-]\d{2}[./-]\d{2,4})\s+(\d{2}:\d{2}\s*[AP]M)?\s+(?:<DIR>|[\d,]+|)\s+(.*)$', clean_line, re.I)
                if match:
                    date_part = match.group(1)
                    time_part = match.group(2) or ""
                    file_part = match.group(3).strip()
                    
                    if file_part in ['.', '..']: continue # Skip directory navigation
                    
                    results.append({
                        'host_id': host_id, 'source_file': file_path, 'evidence_source': mode.upper(),
                        'execution_timestamp': pd.to_datetime(f"{date_part} {time_part}".strip(), errors='coerce'),
                        'executable_path': file_part
                    })
        return results
    except:
        return []

def parse_shimcache_paths(file_path, host_id):
    """Extracts executable paths from ShimCache registry dumps."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        paths = re.findall(r'\\??\\([a-zA-Z]:\\.*?\.exe)', content)
        for p in set(paths):
            results.append({
                'host_id': host_id, 'source_file': file_path, 'evidence_source': 'ShimCache',
                'executable_path': p, 'execution_timestamp': pd.NaT
            })
        return results
    except: return []

def parse_muicache_dump(file_path, host_id):
    """Parses MuiCache to map executables to friendly names."""
    results = []
    try:
        with open(file_path, 'r', encoding='cp437', errors='ignore') as f:
            content = f.read()
        matches = re.findall(r'^\s+(.*?)\.(FriendlyAppName|ApplicationCompany)\s+REG_SZ\s+(.*)$', content, re.MULTILINE)
        temp_map = {}
        for path, attr, val in matches:
            if path not in temp_map: temp_map[path] = {'host_id': host_id, 'source_file': file_path, 'executable_path': path}
            if attr == 'FriendlyAppName': temp_map[path]['friendly_name'] = val.strip()
            if attr == 'ApplicationCompany': temp_map[path]['publisher'] = val.strip()
        return list(temp_map.values())
    except: return []

def parse_ot_hardware_csv(file_path, host_id):
    """Robust WMIC CSV parser for quoted paths (re-included for process_paths)."""
    results = []
    try:
        with open(file_path, 'rb') as f:
            raw_bin = f.read().replace(b'\x00', b'')
        content = None
        for enc in ['utf-16', 'utf-8', 'cp1251']:
            try:
                content = raw_bin.decode(enc, errors='ignore')
                if 'Node' in content: break
            except: continue
        if not content: return []
        reader = csv.reader(io.StringIO(content.strip()))
        raw_headers = next(reader, None)
        if not raw_headers: return []
        headers = [re.sub(r'[^\w]', '', h).lower() for h in raw_headers]
        for row in reader:
            if not row: continue
            item = {headers[i]: row[i].strip().strip('"') for i in range(len(headers)) if i < len(row)}
            item['host_id'] = host_id
            item['source_file'] = file_path
            results.append(item)
        return results
    except: return []

#### Main Loop

In [ ]:
execution_timeline = []
muicache_list = []
volatile_files = []
process_paths_list = [] # This must be defined here to avoid NameError

In [ ]:
phase10_targets = df_file_map[df_file_map['data_category'] == 'volatile_forensics']
unique_hosts = phase10_targets['host_id'].unique()

In [ ]:
for h_id in tqdm(unique_hosts, desc="Parsing Volatile Evidence"):
    host_files = phase10_targets[phase10_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path, f_name = row['full_path'], row['file_name'].lower()
        
        # 1. Timeline Evidence
        if 'userassist' in f_name:
            data = parse_userassist_dump(f_path, h_id)
            for item in data: item['hostname'] = h_name; execution_timeline.append(item)
        elif 'bam_activity' in f_name:
            data = parse_bam_dump(f_path, h_id)
            for item in data: item['hostname'] = h_name; execution_timeline.append(item)
        elif 'prefetch_list' in f_name:
            data = parse_volatile_list(f_path, h_id, mode='Prefetch')
            for item in data: item['hostname'] = h_name; execution_timeline.append(item)
        elif 'shimcache' in f_name:
            data = parse_shimcache_paths(f_path, h_id)
            for item in data: item['hostname'] = h_name; execution_timeline.append(item)
            
        # 2. Metadata & Files (Recent & Temp)
        elif 'muicache' in f_name:
            data = parse_muicache_dump(f_path, h_id)
            for item in data: item['hostname'] = h_name; muicache_list.append(item)
        elif 'recent_files' in f_name or 'temp_contents' in f_name:
            mode = 'Recent' if 'recent' in f_name else 'Temp'
            data = parse_volatile_list(f_path, h_id, mode=mode)
            for item in data: item['hostname'] = h_name; volatile_files.append(item)
            
        # 3. Process Paths
        elif 'process_paths' in f_name:
            data = parse_ot_hardware_csv(f_path, h_id)
            for item in data:
                process_paths_list.append({
                    'host_id': h_id, 'hostname': h_name, 'source_file': f_path,
                    'pid': item.get('processid'), 'name': item.get('name'),
                    'executable_path': item.get('executablepath')
                })

In [ ]:
df_volatile_execution_final = pd.DataFrame(execution_timeline)
df_volatile_muicache_final = pd.DataFrame(muicache_list)
df_volatile_files_final = pd.DataFrame(volatile_files)
df_process_paths_final = pd.DataFrame(process_paths_list)

### OT/ICS Registry Artifacts
---
**Overview**
This phase audits the "Industrial DNA" of the system by analyzing registry configurations related to industrial communication protocols and vendor-specific software footprints. We focus on **OPC (OLE for Process Control)**, **DCOM** security limits, and **COM port** parameters, which are the backbone of legacy ICS integration. Additionally, we scan for the presence of major ICS vendors (Siemens, ABB, Emerson, etc.) and inventory device class GUIDs to identify specialized industrial hardware adapters.

**Target Artifacts:**
* `com_port_config.txt`: Serial port parameters (Baud rate, Parity) critical for Modbus/Serial comms.
* `dcom_config.txt`: Distributed COM security settings (the primary cause of OPC connectivity issues).
* `rpc_config.txt`: Remote Procedure Call settings and endpoint mapper configuration.
* `ics_software.txt`: Registry markers for known ICS/SCADA vendors.
* `opc_servers.txt`: Registered OPC servers and their associated CLSIDs.
* `device_classes.txt`: Inventory of all registered device class GUIDs.

**Output Datasets & Mapping:**

1. **`df_ot_comm_config_final` (Industrial Communication)**
   * *Sources:* `com_port_config.txt`, `dcom_config.txt`, `rpc_config.txt`.
2. **`df_ot_vendor_presence_final` (ICS Footprint)**
   * *Source:* `ics_software.txt`.
3. **`df_opc_inventory_final` (OPC Server Registry)**
   * *Source:* `opc_servers.txt`.
4. **`df_device_classes_final` (Hardware Classes)**
   * *Source:* `device_classes.txt`.

**Data Schema (df_ot_comm_config_final):**

| Field | Type | Description |
| :--- | :--- | :--- |
| `host_id` | string | Unique key: `location__system__hostname`. |
| `hostname` | string | Cleaned computer name. |
| `config_type` | string | Category (e.g., DCOM, RPC, Serial). |
| `registry_key` | string | Full path to the configuration key. |
| `parameter_name` | string | Name of the setting (e.g., MachineAccessRestriction). |
| `parameter_value`| string | Configured value (normalized). |


#### Local Functions

In [ ]:
def parse_registry_dump_recursive(file_path, host_id):
    """
    Universal parser for 'reg query /s' output.
    Groups values under their respective HKEY paths.
    """
    results = []
    try:
        if os.path.getsize(file_path) == 0:
            return []
            
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
            
        # Split by HKEY headers
        blocks = re.split(r'\n(HKEY_.*?)\n', content)
        
        # blocks[0] is usually empty or junk, then pairs of (key, values)
        for i in range(1, len(blocks), 2):
            key_path = blocks[i].strip()
            values_block = blocks[i+1]
            
            # Find all Value-Type-Data triplets
            # Pattern: 4 spaces, Name, Type (REG_...), Data
            matches = re.findall(r'^\s{4}(?P<name>.*?)\s+(?P<type>REG_[A-Z_]+)\s+(?P<data>.*)$', values_block, re.MULTILINE)
            
            for m in matches:
                results.append({
                    'host_id': host_id,
                    'registry_key': key_path,
                    'parameter_name': m[0].strip(),
                    'parameter_type': m[1].strip(),
                    'parameter_value': m[2].strip(),
                    'source_file': file_path
                })
        return results
    except:
        return []

def classify_ot_config_type(file_name):
    """Categorizes the registry artifact based on its filename."""
    fn = file_name.lower()
    if 'com_port' in fn: return 'Serial/COM'
    if 'dcom' in fn: return 'DCOM/Ole'
    if 'rpc' in fn: return 'RPC'
    if 'ics_software' in fn: return 'ICS Vendor Marker'
    if 'opc_servers' in fn: return 'OPC Server'
    if 'device_classes' in fn: return 'Hardware Class'
    return 'Other Registry'

#### Main Loop

In [ ]:
ot_comm_list = []
device_classes_list = []
vendor_presence_list = []

In [ ]:
phase11_targets = df_file_map[df_file_map['data_category'] == 'ot_registry']
unique_hosts = phase11_targets['host_id'].unique()


In [ ]:
for h_id in tqdm(unique_hosts, desc="Parsing OT Registry"):
    host_files = phase11_targets[phase11_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        # Parse the raw registry dump
        reg_data = parse_registry_dump_recursive(f_path, h_id)
        if not reg_data: continue
        
        config_type = classify_ot_config_type(f_name)
        
        for item in reg_data:
            item['hostname'] = h_name
            item['config_category'] = config_type
            
            # Route data to specific datasets
            if config_type in ['Serial/COM', 'DCOM/Ole', 'RPC']:
                ot_comm_list.append(item)
            
            elif config_type == 'Hardware Class':
                # For device classes, we keep the full structure
                device_classes_list.append(item)
                
            elif config_type in ['ICS Vendor Marker', 'OPC Server']:
                vendor_presence_list.append(item)

In [ ]:
# --- Create Final Datasets ---
df_ot_comm_config_final = pd.DataFrame(ot_comm_list)
df_device_classes_final = pd.DataFrame(device_classes_list)
df_ot_vendor_presence_final = pd.DataFrame(vendor_presence_list)

In [ ]:
df_ot_comm_config_final.head()

In [ ]:
df_device_classes_final.head()

In [ ]:
df_ot_vendor_presence_final.head()

### Remote Access Inventory
---
**Overview**
This phase audits the remote management surface and connectivity history of the host. We analyze configurations for native protocols (**RDP**, **WinRM**) and third-party tools (**VNC**, **AnyDesk**, **TeamViewer**). A key component of this audit is the **RDP Connection History**, which reveals outbound connections and associated usernames, providing essential evidence for mapping administrative movement across the ICS network.

**Target Artifacts:**
* `rdp_config.txt`: RDP service state, port, and NLA settings.
* `rdp_history.txt`: Client-side history of remote systems (MRU and Server lists).
* `winrm_config.txt`: WinRM security settings (Authentication, TrustedHosts).
* `vnc_config.txt`, `teamviewer.txt`, `anydesk.txt`: Registry markers for 3rd-party tools.
* `ssh_config.txt` & `sshd_service.txt`: OpenSSH server configuration and status.
* `remote_registry.txt`: Status of the Remote Registry service.

**Output Datasets:**
1. **`df_remote_access_config_final`**: Consolidated settings for RDP, WinRM, and SSH.
2. **`df_rdp_history_final`**: Outbound connection history (Target IP, Username).
3. **`df_remote_tools_presence_final`**: Inventory of installed 3rd-party remote access software.


#### Local Functions

In [ ]:
def parse_rdp_history(file_path, host_id):
    """
    Parses RDP connection history from registry dumps.
    Extracts target hosts from 'Default' (MRU) and 'Servers' keys.
    """
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        
        # 1. Extract MRU (Most Recently Used) list from 'Default' key
        # Pattern: MRU0 REG_SZ 192.168.1.1
        mru_matches = re.findall(r'Terminal Server Client\\Default\n(?:\s+.*\n)*?\s+MRU\d+\s+REG_SZ\s+(.*)', content)
        for target in mru_matches:
            results.append({
                'host_id': host_id, 'source_file': file_path, 'evidence_type': 'RDP_MRU',
                'target_host': target.strip(), 'username_hint': None
            })
            
        # 2. Extract Server-specific settings (includes UsernameHint)
        # Split by the 'Servers' subkeys
        server_blocks = re.split(r'\n(HKEY_.*?Terminal Server Client\\Servers\\.*?)\n', content)
        for i in range(1, len(server_blocks), 2):
            key_path = server_blocks[i].strip()
            properties = server_blocks[i+1]
            
            target_host = key_path.split('\\')[-1]
            user_match = re.search(r'UsernameHint\s+REG_SZ\s+(.*)', properties)
            
            results.append({
                'host_id': host_id, 'source_file': file_path, 'evidence_type': 'RDP_Server_Entry',
                'target_host': target_host, 'username_hint': user_match.group(1).strip() if user_match else None
            })
            
        return results
    except:
        return []

def parse_registry_dump_simple(file_path, host_id):
    """
    Universal parser for 'reg query' output. 
    Reused for RDP, WinRM and Tool configs.
    """
    results = []
    try:
        if os.path.getsize(file_path) == 0: return []
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        blocks = re.split(r'\n(HKEY_.*?)\n', content)
        for i in range(1, len(blocks), 2):
            key_path = blocks[i].strip()
            values_block = blocks[i+1]
            matches = re.findall(r'^\s{4}(?P<name>.*?)\s+(?P<type>REG_[A-Z_]+)\s+(?P<data>.*)$', values_block, re.MULTILINE)
            for m in matches:
                results.append({
                    'host_id': host_id, 'registry_key': key_path,
                    'parameter_name': m[0].strip(), 'parameter_type': m[1].strip(),
                    'parameter_value': m[2].strip(), 'source_file': file_path
                })
        return results
    except: return []

def parse_sc_status_simple(file_path, host_id):
    """Parses sc query output for service state."""
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        if "SERVICE_NAME" not in content: return []
        
        s_name = re.search(r'SERVICE_NAME:\s+(.*)', content)
        state = re.search(r'STATE\s+:\s+\d+\s+(.*)', content)
        
        return [{
            'host_id': host_id, 'service_name': s_name.group(1).strip() if s_name else "Unknown",
            'state': state.group(1).strip() if state else "Unknown",
            'source_file': file_path
        }]
    except: return []

#### Main Loop


In [ ]:
remote_config_list = []
rdp_history_list = []
remote_tools_list = []

In [ ]:
phase12_targets = df_file_map[df_file_map['data_category'] == 'remote_access']
unique_hosts = phase12_targets['host_id'].unique()


In [ ]:
for h_id in tqdm(unique_hosts, desc="Auditing Remote Access"):
    host_files = phase12_targets[phase12_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        # 1. RDP History
        if 'rdp_history' in f_name:
            data = parse_rdp_history(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                rdp_history_list.append(item)
        
        # 2. Protocol Configurations (RDP, WinRM, SSH)
        elif any(x in f_name for x in ['rdp_config', 'winrm_config', 'ssh_config']):
            data = parse_registry_dump_simple(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                item['config_type'] = 'RDP' if 'rdp' in f_name else ('WinRM' if 'winrm' in f_name else 'SSH')
                remote_config_list.append(item)
                
        # 3. Third-party Tools
        elif any(x in f_name for x in ['vnc', 'teamviewer', 'anydesk']):
            data = parse_registry_dump_simple(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                item['tool_name'] = f_name.split('_')[0].upper()
                remote_tools_list.append(item)
                
        # 4. Service Status
        elif any(x in f_name for x in ['sshd_service', 'remote_registry']):
            data = parse_sc_status_simple(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                item['config_type'] = 'Service_Status'
                remote_config_list.append(item)


In [ ]:
df_remote_access_config_final = pd.DataFrame(remote_config_list)
df_rdp_history_final = pd.DataFrame(rdp_history_list).drop_duplicates(['host_id', 'target_host', 'username_hint'])
df_remote_tools_presence_final = pd.DataFrame(remote_tools_list)


###  Integrity Markers

**Overview**
This phase establishes a security baseline by auditing the integrity of critical system binaries and configuration files. By calculating SHA256 hashes of essential executables (like `lsass.exe`, `svchost.exe`, `cmd.exe`) and the `hosts` file, we can detect unauthorized modifications, file replacements, or DNS redirection attacks. Additionally, we analyze the **Component Based Servicing (CBS)** logs to identify historical records of system file corruption and automatic repairs performed by Windows.

**Target Artifacts:**
* `system_hashes.txt`: SHA256 hashes of core Windows binaries generated via `certutil`.
* `hosts_hash.txt`: SHA256 hash of the `%SystemRoot%\system32\drivers\etc\hosts` file.
* `cbs.log`: Windows servicing log containing integrity check and repair history.
* `sfc_note.txt`: Status marker for System File Checker (SFC) availability.

**Output Datasets & Mapping:**

1. **`df_integrity_hashes_final` (Binary Fingerprints)**
   * *Sources:* `system_hashes.txt`, `hosts_hash.txt`.
2. **`df_integrity_status_final` (Repair History)**
   * *Sources:* `cbs.log`, `sfc_note.txt`.



#### Data schema



| Field | Type | Description |
| :--- | :--- | :--- |
| `host_id` | string | Unique key: `location__system__hostname`. |
| | `file_path` | string | Full path to the audited system file. |
| | `hash_sha256` | string | Calculated SHA256 signature. |
| | `is_known_good` | boolean | Placeholder for comparison against a trusted baseline. |



#### Local Functions

In [ ]:
def parse_certutil_hashes(file_path, host_id):
    """
    Parses 'certutil -hashfile' output. 
    Handles multiple files and single file (hosts) formats.
    """
    results = []
    try:
        with open(file_path, 'r', encoding='cp1251', errors='ignore') as f:
            content = f.read()
        
        # Split by "File:" marker to handle system_hashes.txt
        # If not found, treat as a single hash file (hosts_hash.txt)
        if "File:" in content:
            blocks = content.split('File: ')
        else:
            blocks = [content]

        for block in blocks:
            if not block.strip(): continue
            
            # Extract path: handles quoted paths like "C:\Windows\..."
            path_match = re.search(r'["\']?([a-zA-Z]:\\[^"\n\r:]+)["\']?', block)
            # Extract 64-char SHA256 hex string
            hash_match = re.search(r'\b([a-fA-F0-9]{64})\b', block)
            
            if hash_match:
                results.append({
                    'host_id': host_id,
                    'file_path': path_match.group(1).strip() if path_match else "Unknown Path",
                    'hash_sha256': hash_match.group(1).lower(),
                    'source_file': file_path
                })
        return results
    except:
        return []

def analyze_cbs_log_summary(file_path, host_id):
    """
    Scans CBS.log for corruption, repairs, and pending reboots.
    """
    results = []
    try:
        corruption_count = 0
        repair_count = 0
        reboot_pending = False
        
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                # Forensic markers
                if "CSI Payload Corrupt" in line or "corrupt" in line.lower():
                    corruption_count += 1
                if "Repairing corrupted file" in line or "Fixed" in line:
                    repair_count += 1
                if "RebootPending" in line and "volatile key indicates" in line:
                    reboot_pending = True
        
        results.append({
            'host_id': host_id,
            'corruption_events': corruption_count,
            'repair_events': repair_count,
            'reboot_pending': reboot_pending,
            'status': 'Issues Detected' if corruption_count > 0 else 'Healthy',
            'source_file': file_path
        })
        return results
    except:
        return []

#### Main Loop


In [ ]:
integrity_hashes_list = []
integrity_status_list = []

In [ ]:
phase13_targets = df_file_map[df_file_map['data_category'] == 'integrity_markers']
unique_hosts = phase13_targets['host_id'].unique()


In [ ]:
for h_id in tqdm(unique_hosts, desc="Auditing System Integrity"):
    host_files = phase13_targets[phase13_targets['host_id'] == h_id]
    h_name = host_files.iloc[0]['hostname']
    
    for _, row in host_files.iterrows():
        f_path = row['full_path']
        f_name = row['file_name'].lower()
        
        # 1. Process Hashes (System binaries and Hosts file)
        if 'hash' in f_name:
            data = parse_certutil_hashes(f_path, h_id)
            for item in data:
                item['hostname'] = h_name
                integrity_hashes_list.append(item)
        
        # 2. Process CBS Logs
        elif f_name == 'cbs':
            data = analyze_cbs_log_summary(f_path, h_id) # Name fixed
            for item in data:
                item['hostname'] = h_name
                integrity_status_list.append(item)
        
        # 3. Process SFC Notes
        elif f_name == 'sfc_note':
            try:
                with open(f_path, 'r') as f: note = f.read().strip()
                integrity_status_list.append({
                    'host_id': h_id, 'hostname': h_name, 'status': 'SFC Info',
                    'details': note, 'source_file': f_path,
                    'corruption_events': 0, 'repair_events': 0, 'reboot_pending': False
                })
            except: pass

In [ ]:
df_integrity_hashes_final = pd.DataFrame(integrity_hashes_list)
df_integrity_status_final = pd.DataFrame(integrity_status_list)


## Stage 3: Master Host Table Assembly

**Overview**
The **Master Host Table (`df_master_host`)** is the primary analytical engine of this project. It consolidates the most critical metrics and security indicators from all 13 collection phases into a single, high-density record per host. This "Golden Record" allows for instantaneous cross-domain analysis—such as correlating OS age with open network ports or administrative privilege density. By transforming 30+ fragmented datasets into one unified view, we enable rapid risk assessment and automated compliance auditing across the entire ICS environment.

**Key Objectives:**
1.  **Centralized Identity:** Establish a definitive baseline for every host, including its physical location, functional system, and vendor.
2.  **Multi-Phase Enrichment:** Aggregate hardware specs, network footprints, security postures, and forensic markers into a single row.
3.  **Risk Indicator Derivation:** Calculate high-level flags such as `is_legacy`, `is_virtual`, and `protection_gap` based on consolidated data.
4.  **Data Integrity:** Ensure 100% host coverage by using `left join` operations, preserving nodes even if specific phase data is missing.

**Target Features for the Master Table:**

*   **Identity:** `location`, `system`, `system_category`, `vendor`, `node_type`.
*   **Technical Passport:** `os_name`, `os_version`, `ram_total_gb`, `cpu_cores_count`, `serial_number`.
*   **Network Footprint:** `primary_ip`, `primary_mac`, `listening_ports_count`, `established_connections_count`.
*   **Security & Compliance:** `antivirus_status`, `total_patches_count`, `last_patch_date`, `admin_accounts_count`.
*   **Storage Health:** `total_disk_gb`, `min_free_space_pct`.
*   **Forensic Markers:** `usb_devices_detected_count`, `critical_security_events_count`.



In [ ]:
# --- 1. Initialize Base Table ---
# We take unique hosts and their primary metadata from Stage 1
df_master_host = df_file_map.drop_duplicates('host_id')[
    ['host_id', 'hostname', 'location', 'system', 'system_category', 'vendor', 'node_type']
].copy()



In [ ]:
# --- 2. Integrate OS and Hardware Passport ---
if 'df_os_hw_final' in globals():
    passport_cols = [
        'host_id', 'os_name', 'os_version', 'os_install_date', 
        'last_boot_time', 'ram_total_gb', 'cpu_cores_count', 'serial_number'
    ]
    # Filter only existing columns to avoid merge errors
    existing_passport_cols = [c for c in passport_cols if c in df_os_hw_final.columns]
    df_master_host = df_master_host.merge(df_os_hw_final[existing_passport_cols], on='host_id', how='left')



In [ ]:
# --- 3. Integrate Network Footprint ---
if 'df_net_interfaces_final' in globals() and not df_net_interfaces_final.empty:
    # Get primary IP and MAC (first active interface found)
    net_basic = df_net_interfaces_final.dropna(subset=['ipv4_address']).groupby('host_id').agg({
        'ipv4_address': 'first',
        'mac_address': 'first'
    }).reset_index().rename(columns={'ipv4_address': 'primary_ip', 'mac_address': 'primary_mac'})
    df_master_host = df_master_host.merge(net_basic, on='host_id', how='left')

if 'df_net_connections_final' in globals() and not df_net_connections_final.empty:
    # Count listening ports and established connections
    conn_stats = df_net_connections_final.groupby('host_id').agg({
        'state': [lambda x: (x == 'LISTENING').sum(), lambda x: (x == 'ESTABLISHED').sum()]
    }).reset_index()
    conn_stats.columns = ['host_id', 'listening_ports_count', 'established_conns_count']
    df_master_host = df_master_host.merge(conn_stats, on='host_id', how='left')



In [ ]:
# --- 4. Integrate Security Posture ---
if 'df_av_status_final' in globals() and not df_av_status_final.empty:
    av_summary = df_av_status_final.groupby('host_id').agg({
        'av_product_name': 'first',
        'av_product_state': 'first'
    }).reset_index()
    df_master_host = df_master_host.merge(av_summary, on='host_id', how='left')

if 'df_group_members_final' in globals() and not df_group_members_final.empty:
    # Count local administrators
    admin_counts = df_group_members_final[
        df_group_members_final['group_name'].str.contains('Administrators', case=False, na=False)
    ].groupby('host_id').size().reset_index(name='admin_accounts_count')
    df_master_host = df_master_host.merge(admin_counts, on='host_id', how='left')

if 'df_patches_final' in globals() and not df_patches_final.empty:
    patch_stats = df_patches_final.groupby('host_id').agg({
        'kb_id': 'count',
        'installed_on': 'max'
    }).reset_index().rename(columns={'kb_id': 'total_patches_count', 'installed_on': 'last_patch_date'})
    df_master_host = df_master_host.merge(patch_stats, on='host_id', how='left')



In [ ]:
# --- 5. Integrate Storage Health ---
if 'df_disks_logical_final' in globals() and not df_disks_logical_final.empty:
    disk_stats = df_disks_logical_final.groupby('host_id').agg({
        'size_gb': 'sum',
        'free_space_pct': 'min'
    }).reset_index().rename(columns={'size_gb': 'total_storage_gb', 'free_space_pct': 'min_free_pct'})
    df_master_host = df_master_host.merge(disk_stats, on='host_id', how='left')



In [ ]:
# --- 6. Integrate Forensic Markers ---
if 'df_usb_history_final' in globals() and not df_usb_history_final.empty:
    usb_counts = df_usb_history_final.groupby('host_id').size().reset_index(name='usb_devices_detected_count')
    df_master_host = df_master_host.merge(usb_counts, on='host_id', how='left')

if 'df_event_logs_critical_final' in globals() and not df_event_logs_critical_final.empty:
    event_counts = df_event_logs_critical_final.groupby('host_id').size().reset_index(name='critical_events_count')
    df_master_host = df_master_host.merge(event_counts, on='host_id', how='left')



In [ ]:
# --- 7. Feature Engineering: Risk Indicators ---
# A. Legacy OS Flag
def check_legacy(os_name):
    os_str = str(os_name).upper()
    return any(x in os_str for x in ['XP', '2003', '2000', 'NT'])
df_master_host['is_legacy'] = df_master_host['os_name'].apply(check_legacy)

# B. Virtualization Flag
def check_virtual(vendor, model):
    vm_str = (str(vendor) + " " + str(model)).upper()
    return any(x in vm_str for x in ['VMWARE', 'VIRTUAL', 'HYPER-V', 'XEN', 'QEMU'])
df_master_host['is_virtual'] = df_master_host.apply(lambda x: check_virtual(x.get('vendor'), x.get('model')), axis=1)

# C. Protection Gap Flag (Modern OS with no AV)
df_master_host['protection_gap'] = (df_master_host['is_legacy'] == False) & (df_master_host['av_product_name'].isna())


In [ ]:
# --- 8. Final Cleanup and Normalization ---
pd.set_option('future.no_silent_downcasting', True)

# Fill numeric gaps with 0
count_cols = [
    'listening_ports_count', 'established_conns_count', 'admin_accounts_count', 
    'total_patches_count', 'usb_devices_detected_count', 'critical_events_count',
    'ram_total_gb', 'cpu_cores_count', 'total_storage_gb'
]
for col in count_cols:
    if col in df_master_host.columns:
        df_master_host[col] = df_master_host[col].fillna(0)

# Fill string gaps
df_master_host['av_product_name'] = df_master_host['av_product_name'].fillna('None Detected')
df_master_host['os_name'] = df_master_host['os_name'].fillna('Unknown OS')


In [ ]:
df_master_host.info()